# Minesweeper Agent Minesweeper Agent

![Minesweeper Agent Flow](images/minesweeper_agent_flow.png)

## Overview

Welcome! In this notebook we are going to do two things together, in order:

1. **Design the game**: we hand the AI a deliberately broken Minesweeper spec and watch it observe, plan, act, and reflect until the spec satisfies our constraints.
2. **Play the game**: once the spec is fixed, the SAME agent picks up the resulting toolset and actually plays Minesweeper, one click at a time, with you driving each turn via a `[Next Move →]` button.

By the end you should be able to point at any decision the agent makes and say exactly what it observed, what it chose, which tool it called, and how it knew the move worked.

Here is what we will cover, in order:

- **Agentic AI fundamentals** — how a system that works toward a goal differs from one that only produces text.
- **Tool use** — how the agent reaches out and calls plain Python functions.
- **A reasoning trace** — the visible record of every decision, so nothing is hidden inside a black box.
- **Human approval** — why a safe agent pauses before applying an important change.
- **A full LM Studio loop** — model discovery via `GET /v1/models`, JSON tool calls, transparent fallback.
- **The spec → play causality** — if you cut a feature in design, the playing agent loses that tool. This is the most important moment of the case.

This notebook assumes LM Studio is running on your machine with at least one chat model loaded. Step 2.5 verifies that before any LLM work.

## How to Read This Notebook

I have written this as a standalone introduction to Agentic AI — you do not need to have read any other case first. We move in small steps, and every step shows its result before we add the next idea. Route:

1. We meet **Minesweeper** itself: 30 seconds of rules and one demo board.
2. We try a move by hand so you feel the game is real.
3. We learn what an Agent is (using Minesweeper as the running example).
4. We ask LM Studio which models it has loaded and pick a default.
5. We define the **spec** — a Python dictionary describing the game we want.
6. We compare three solvers on the SAME spec-fixing job: fixed rule, chatbot, agent.
7. We trace the agent's decision so you read *why*, not just *what*.
8. We hand the spec-fixing loop to LM Studio for real.
9. We hand the resulting toolset to the agent, and **watch it play Minesweeper** one click at a time.

One sentence I want you to walk away with:

```text
state + tools + feedback + revision = agentic behavior
```

If you can spot those four pieces in any agent demo later, you are already ahead of most tutorials.

## What is Minesweeper?

In case you have never played: Minesweeper is a single-player puzzle on a small grid (we use 6×6) with a handful of hidden mines (we use 6). Your view is mostly hidden cells. Each turn you can:

- **reveal** a cell — if it is a mine, the game ends; otherwise it shows a number `K` meaning "exactly K of my 8 neighbours are mines".
- **flag** a cell — mark "I think a mine is here" so you do not accidentally reveal it.
- **unflag** a cell — change your mind.

You **win** when every non-mine cell is revealed. You **lose** the instant you reveal a mine — the whole board flips over to show where the mines were, and the score is final.

Reasoning happens like this: a revealed `1` next to a single hidden cell tells you that cell is definitely a mine (flag it). A revealed `2` with two flagged neighbours tells you the other hidden neighbours are safe (reveal them). When no deduction works, you guess — and guesses are usually safer in corners than in the middle.

Here is what a partly-played board looks like. We will use this rendering all the way through:

In [ ]:
"""Game-design tools you (and later the Agent) can call.

Each tool is an ordinary, deterministic Python function. The agentic behaviour
does not come from these tools — it comes from deciding which one to call
after looking at the current game spec. The tools just do exactly what they
say.

The state we operate on is a "game spec": a dict that describes what the
finished Minesweeper game should look like — which actions the player is
allowed to take, which hints are shown, and which safety conveniences are
turned on. Tools either *read* the spec (the check_* functions) or return
a *revised* spec (remove_feature, simplify_feature, add_feature,
reorder_feature). They never mutate the spec in place — that way you can
always compare "before" and "after".
"""
from copy import deepcopy
from typing import Any
Spec = dict[str, Any]
ToolResult = dict[str, Any]
FEATURE_NAME_TO_ACTION = {'Reveal cell': 'reveal', 'Flag cell': 'flag', 'Unflag cell': 'unflag'}

def copy_spec(spec: Spec) -> Spec:
    """Return a deep copy so tool calls do not mutate earlier observations."""
    return deepcopy(spec)

def create_initial_spec() -> Spec:
    """Create an intentionally imperfect Minesweeper game spec.

    The spec exceeds the complexity_limit on purpose AND is missing one of
    the required feature types (``safety``), so you can watch the Agent
    detect both problems and repair them.
    """
    return {'title': 'Minesweeper Easy', 'audience': 'freshmen learning Python', 'board_size': 6, 'mine_count': 6, 'complexity_limit': 5, 'must_include': ['action', 'hint', 'safety'], 'features': [{'name': 'Reveal cell', 'complexity': 1, 'type': 'action', 'required': True, 'status': 'active'}, {'name': 'Flag cell', 'complexity': 1, 'type': 'action', 'required': True, 'status': 'active'}, {'name': 'Unflag cell', 'complexity': 1, 'type': 'action', 'required': False, 'status': 'active'}, {'name': 'Numeric hints (1-8)', 'complexity': 1, 'type': 'hint', 'required': True, 'status': 'active'}, {'name': 'Flood-fill on zero', 'complexity': 2, 'type': 'qol', 'required': False, 'status': 'active'}, {'name': 'Auto-flag completed', 'complexity': 3, 'type': 'qol', 'required': False, 'status': 'active'}, {'name': 'Mine counter HUD', 'complexity': 2, 'type': 'ui', 'required': False, 'status': 'active'}], 'engine': 'python-stdlib'}

def active_features(spec: Spec) -> list[dict[str, Any]]:
    """Return only the features that still count toward the game."""
    return [item for item in spec['features'] if item.get('status') == 'active']

def total_complexity(spec: Spec) -> int:
    """Sum the complexity of every active feature."""
    return sum((item['complexity'] for item in active_features(spec)))

def check_complexity(spec: Spec) -> ToolResult:
    """Check whether the active features fit within the complexity budget."""
    total = total_complexity(spec)
    limit = spec['complexity_limit']
    return {'passed': total <= limit, 'total': total, 'limit': limit, 'message': 'complexity OK' if total <= limit else f'complexity {total} exceeds limit {limit}'}

def check_requirements(spec: Spec) -> ToolResult:
    """Check whether the spec still includes every required feature type."""
    active_types = {item['type'] for item in active_features(spec)}
    missing = [item for item in spec['must_include'] if item not in active_types]
    return {'passed': not missing, 'missing': missing, 'message': 'requirements OK' if not missing else f'missing required types: {missing}'}

def remove_feature(spec: Spec, name: str) -> tuple[Spec, ToolResult]:
    """Mark a feature as removed and return the revised spec."""
    new_spec = copy_spec(spec)
    for item in new_spec['features']:
        if item['name'] == name and item['status'] == 'active':
            item['status'] = 'removed'
            return (new_spec, {'success': True, 'message': f'removed {name}'})
    return (new_spec, {'success': False, 'message': f'feature not found or inactive: {name}'})

def simplify_feature(spec: Spec, name: str, points: int) -> tuple[Spec, ToolResult]:
    """Reduce a feature's complexity while keeping a 1-point minimum."""
    new_spec = copy_spec(spec)
    for item in new_spec['features']:
        if item['name'] == name and item['status'] == 'active':
            old = item['complexity']
            item['complexity'] = max(1, old - points)
            return (new_spec, {'success': True, 'message': f'simplified {name} from {old} to {item['complexity']}'})
    return (new_spec, {'success': False, 'message': f'feature not found or inactive: {name}'})

def add_feature(spec: Spec, name: str, complexity: int, feature_type: str, required: bool=False) -> tuple[Spec, ToolResult]:
    """Append a new feature to the game spec."""
    new_spec = copy_spec(spec)
    new_spec['features'].append({'name': name, 'complexity': complexity, 'type': feature_type, 'required': required, 'status': 'active'})
    return (new_spec, {'success': True, 'message': f'added {name}'})

def reorder_feature(spec: Spec, name: str, new_index: int) -> tuple[Spec, ToolResult]:
    """Move a feature to a new index in the feature list."""
    new_spec = copy_spec(spec)
    features = new_spec['features']
    for index, item in enumerate(features):
        if item['name'] == name:
            moved = features.pop(index)
            bounded_index = max(0, min(new_index, len(features)))
            features.insert(bounded_index, moved)
            return (new_spec, {'success': True, 'message': f'moved {name} to position {bounded_index}'})
    return (new_spec, {'success': False, 'message': f'feature not found: {name}'})

def derive_allowed_actions(spec: Spec) -> list[str]:
    """Translate the active 'action' features into the agent's tool list for Phase B.

    Only features whose name appears in FEATURE_NAME_TO_ACTION are translated,
    and only while they are still active. So if Phase A removes "Flag cell",
    the Phase B agent literally cannot flag anything.
    """
    out: list[str] = []
    for item in active_features(spec):
        action = FEATURE_NAME_TO_ACTION.get(item['name'])
        if action and action not in out:
            out.append(action)
    return out

def generate_final_report(spec: Spec) -> ToolResult:
    """Summarise the final game spec so the report cell can print it."""
    complexity = check_complexity(spec)
    requirements = check_requirements(spec)
    return {'title': spec['title'], 'audience': spec['audience'], 'board_size': spec.get('board_size'), 'mine_count': spec.get('mine_count'), 'total_complexity': complexity['total'], 'complexity_limit': complexity['limit'], 'complexity_passed': complexity['passed'], 'requirements_passed': requirements['passed'], 'active_features': active_features(spec), 'allowed_actions_for_play': derive_allowed_actions(spec), 'engine': spec['engine']}
'A small, inspectable Agent loop you can read end to end.\n\nWe deliberately do not use any Agent framework here. The point of this case\nis that you can see the whole loop in plain Python: observe the spec,\npropose a few candidate actions, pick one, run a tool, then reflect on the\nresult. Once you understand this skeleton,\n``utils.llm_client.llm_agent_loop`` swaps the "propose + pick" step for a\nreal LM Studio call so you can watch an LLM drive the same loop.\n'
from typing import Any
Action = dict[str, Any]
Observation = dict[str, Any]
Trace = dict[str, Any]
TOOLS = {'remove_feature': remove_feature, 'simplify_feature': simplify_feature, 'add_feature': add_feature}

def observe(spec: Spec) -> Observation:
    """Observe current spec constraints and return easy-to-read issues."""
    complexity = check_complexity(spec)
    requirements = check_requirements(spec)
    issues: list[str] = []
    if not complexity['passed']:
        issues.append(complexity['message'])
    if not requirements['passed']:
        issues.append(requirements['message'])
    return {'complexity': complexity, 'requirements': requirements, 'issues': issues, 'passed': not issues}

def propose_candidate_actions(spec: Spec, observation: Observation) -> list[Action]:
    """Generate ranked candidate actions from the current observation."""
    if observation['passed']:
        return []
    candidates: list[Action] = []
    missing = observation['requirements'].get('missing', [])
    for missing_type in missing:
        if missing_type == 'action':
            candidates.append({'label': 'Add reveal action', 'tool': 'add_feature', 'args': {'name': 'Reveal cell', 'complexity': 1, 'feature_type': 'action', 'required': True}, 'score': 0.92, 'reason': 'Restores the missing action type so the player can actually move.'})
        elif missing_type == 'hint':
            candidates.append({'label': 'Add numeric hint', 'tool': 'add_feature', 'args': {'name': 'Numeric hints (1-8)', 'complexity': 1, 'feature_type': 'hint', 'required': True}, 'score': 0.88, 'reason': 'Restores the missing hint type so the board still shows neighbour counts.'})
        elif missing_type == 'safety':
            candidates.append({'label': 'Add first-click-safe', 'tool': 'add_feature', 'args': {'name': 'First-click safe', 'complexity': 2, 'feature_type': 'safety', 'required': True}, 'score': 0.86, 'reason': 'Restores the missing safety type so a beginner cannot lose on turn 1 by bad luck.'})
    total = observation['complexity']['total']
    limit = observation['complexity']['limit']
    overflow = total - limit
    if overflow > 0:
        active_optional = [item for item in spec['features'] if item.get('status') == 'active' and (not item.get('required', False))]
        for item in sorted(active_optional, key=lambda f: f['complexity'], reverse=True):
            name = item['name']
            cx = item['complexity']
            candidates.append({'label': f'Remove {name}', 'tool': 'remove_feature', 'args': {'name': name}, 'score': min(0.85, 0.45 + 0.1 * cx), 'reason': f'Drops a non-required feature worth {cx} complexity points in one move.'})
            if cx >= 2:
                candidates.append({'label': f'Simplify {name}', 'tool': 'simplify_feature', 'args': {'name': name, 'points': min(overflow, cx - 1)}, 'score': 0.4 + 0.05 * cx, 'reason': f'Keeps {name} but trims its complexity score so the feature stays without overrunning the budget.'})
    return sorted(candidates, key=lambda item: item['score'], reverse=True)

def choose_action(candidates: list[Action]) -> Action | None:
    """Choose the highest-scoring candidate action."""
    if not candidates:
        return None
    return max(candidates, key=lambda item: item['score'])

def act(spec: Spec, action: Action) -> tuple[Spec, ToolResult]:
    """Execute a selected action by calling its deterministic tool."""
    tool = TOOLS[action['tool']]
    return tool(spec, **action['args'])

def run_agent_step(spec: Spec) -> tuple[Spec, Trace]:
    """Run one observable Observe → Plan → Act → Reflect cycle."""
    observation = observe(spec)
    candidates = propose_candidate_actions(spec, observation)
    action = choose_action(candidates)
    if action is None:
        result = {'success': True, 'message': 'No action needed'}
        return (spec, {'observation': observation, 'candidates': candidates, 'action': None, 'result': result, 'reflection': observe(spec)})
    new_spec, result = act(spec, action)
    reflection = observe(new_spec)
    return (new_spec, {'observation': observation, 'candidates': candidates, 'action': action, 'result': result, 'reflection': reflection})

def run_agent_until_done(spec: Spec, max_steps: int=4, require_approval: bool=False, approval: str='Approve') -> tuple[Spec, list[Trace]]:
    """Run the Agent for a bounded number of steps.

    The approval argument keeps the notebook safe and explicit: in a real
    class you can change it to Modify or Replan before applying a key action.
    """
    current_spec = spec
    traces: list[Trace] = []
    for _ in range(max_steps):
        if observe(current_spec)['passed']:
            break
        next_spec, trace = run_agent_step(current_spec)
        trace['approval'] = approval if require_approval else 'Auto-approved for demo'
        traces.append(trace)
        if require_approval and approval != 'Approve':
            break
        current_spec = next_spec
        if trace['reflection']['passed']:
            break
    return (current_spec, traces)
'Small helpers we use to render an Agent reasoning trace in the notebook.\n\nEach function returns a short, classroom-friendly string so you can show\nstudents exactly what the Agent observed, considered, did, and concluded.\n'
from typing import Any

def format_observation(observation: dict[str, Any]) -> str:
    """Format the observation panel."""
    if observation['passed']:
        return '👀 Observation: all constraints currently pass.'
    issues = '; '.join(observation['issues'])
    return f'👀 Observation: {issues}'

def format_candidates(candidates: list[dict[str, Any]]) -> str:
    """Format candidate actions with scores and reasons."""
    if not candidates:
        return '🧠 Candidate Decisions: no action needed.'
    lines = ['🧠 Candidate Decisions:']
    for index, candidate in enumerate(candidates, start=1):
        label = candidate.get('label', candidate['tool'])
        lines.append(f'  {index}. {label} | score={candidate['score']:.2f} | {candidate['reason']}')
    return '\n'.join(lines)

def format_tool_call(action: dict[str, Any] | None) -> str:
    """Format the selected tool call."""
    if action is None:
        return '🛠 Tool Call: none'
    return f'🛠 Tool Call: {action['tool']}({action['args']})'

def format_reflection(reflection: dict[str, Any]) -> str:
    """Format post-action reflection."""
    if reflection['passed']:
        return '✅ Reflection: all constraints met.'
    return f'🔁 Reflection: {'; '.join(reflection['issues'])}'

def render_trace(trace: dict[str, Any], goal: str='Design a STEM workshop') -> str:
    """Return a complete text trace that works in terminal and Notebook cells."""
    sections = [f'🎯 Goal: {goal}', format_observation(trace['observation']), format_candidates(trace['candidates']), format_tool_call(trace['action']), f'📌 Result: {trace['result']['message']}', format_reflection(trace['reflection'])]
    output = '\n'.join(sections)
    print(output)
    return output

def plan_to_dataframe(plan: dict[str, Any]):
    """Convert the game spec to a Plan Board DataFrame."""
    import pandas as pd
    return pd.DataFrame(plan['features'])

def display_plan_board(plan: dict[str, Any]) -> None:
    """Display the Plan Board inside Jupyter."""
    from IPython.display import display
    display(plan_to_dataframe(plan))
'LM Studio helpers used by this case.\n\nWe use this module for three things:\n\n1. ``list_lm_studio_models`` — call ``GET /v1/models`` to see which models\n   LM Studio currently has loaded. The notebook uses this at the very\n   beginning to pick a default model.\n2. ``chatbot_baseline`` — a single LLM call that only returns advice text.\n   We show it so you can compare a chatbot ("talks about the game") with an\n   Agent ("changes the game spec").\n3. ``llm_agent_loop`` — a real Observe → Plan → Act → Reflect loop that\n   asks the local LLM what to do at each step and then runs the chosen\n   tool. This is the closing demo of the notebook: you actually see the\n   Agent talk to LM Studio multiple times until the game spec satisfies\n   the constraints.\n\nThe notebook assumes LM Studio is running. If something is unreachable in\nthe middle of the agent loop we still emit a fallback trace and finish the\nloop, so a transient hiccup never leaves a half-finished demo on screen.\n'
import json
import re
from typing import Any, Callable
import requests
DEFAULT_LM_STUDIO_BASE = 'http://127.0.0.1:1234/v1'
DEFAULT_LM_STUDIO_ENDPOINT = f'{DEFAULT_LM_STUDIO_BASE}/chat/completions'
DEFAULT_MODELS_ENDPOINT = f'{DEFAULT_LM_STUDIO_BASE}/models'
DEFAULT_MODEL = 'local-model'

def list_lm_studio_models(endpoint: str=DEFAULT_MODELS_ENDPOINT, timeout: int=5) -> list[str]:
    """Ask LM Studio which models are currently loaded.

    Returns a list of model ids. Raises ``RuntimeError`` with a clear
    message if LM Studio is not reachable, so the notebook can guide you
    to start it before going any further.
    """
    try:
        response = requests.get(endpoint, timeout=timeout)
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(f'Could not reach LM Studio at {endpoint}. Start LM Studio and click Developer → Local Server → Start Server before running this cell. (underlying error: {exc})') from exc
    if response.status_code != 200:
        raise RuntimeError(f'LM Studio responded with HTTP {response.status_code} at {endpoint}. Body: {response.text[:200]}')
    data = response.json()
    models = [item['id'] for item in data.get('data', []) if 'id' in item]
    if not models:
        raise RuntimeError('LM Studio is running but no models are loaded. Load any chat-capable instruct model in LM Studio first.')
    return models

def pick_default_model(models: list[str]) -> str:
    """Pick a sensible default model from the list LM Studio gave us.

    Right now we just take the first one — LM Studio usually lists the
    currently-active model first. You can override it in the notebook by
    setting ``MODEL_NAME`` yourself after this call.
    """
    if not models:
        raise ValueError('Cannot pick a default model from an empty list.')
    return models[0]

def call_local_lmstudio(messages: list[dict[str, str]], endpoint: str=DEFAULT_LM_STUDIO_ENDPOINT, model: str=DEFAULT_MODEL, temperature: float=0.2, timeout: int=60) -> str:
    """Send a chat request to an OpenAI-compatible LM Studio endpoint.

    We keep the payload tiny so you can read every field at a glance:
    ``model``, ``messages``, ``temperature``, ``stream``.
    """
    payload = {'model': model, 'messages': messages, 'temperature': temperature, 'stream': False}
    response = requests.post(endpoint, json=payload, timeout=timeout)
    response.raise_for_status()
    data: dict[str, Any] = response.json()
    return data['choices'][0]['message']['content']

def chatbot_baseline(goal: str, model: str=DEFAULT_MODEL, endpoint: str=DEFAULT_LM_STUDIO_ENDPOINT) -> str:
    """Ask the LLM for game-design advice as plain text.

    Use this when you want to show that a chatbot can sound helpful while
    leaving the game spec completely unchanged.
    """
    messages = [{'role': 'system', 'content': 'You are a STEM teaching assistant helping a beginner design a tiny Python Minesweeper game. Give concise design advice in 3-5 short bullet points. Do NOT write code.'}, {'role': 'user', 'content': goal}]
    return call_local_lmstudio(messages, endpoint=endpoint, model=model)
AGENT_SYSTEM_PROMPT = 'You are an Observable Game-Builder Agent for STEM teachers.\n\nYour job at every step: pick ONE tool call that brings a Minesweeper game spec\ncloser to satisfying two constraints, then output it as ONE JSON object on a\nsingle line. Nothing else.\n\n# THE TWO CONSTRAINTS YOU MUST SATISFY\n1. Total complexity of active features must be <= complexity_limit.\n2. Every required type listed in must_include must appear in at least one\n   active feature (status == "active").\n\n# TOOLS YOU CAN CALL (exact names and exact arg keys)\n- remove_feature        args: {"name": <str>}\n- simplify_feature      args: {"name": <str>, "points": <int>}\n- add_feature           args: {"name": <str>, "complexity": <int>, "feature_type": <str>, "required": <bool>}\n- stop                  args: {}\n\nThe arg key for add_feature is "feature_type" (NOT "type"). The arg key for\nsimplify_feature is "points" (NOT "complexity"). Wrong keys will be rejected.\n\n# OUTPUT FORMAT (rigid)\nReply with ONE JSON object on ONE line. No prose. No markdown. No code fences.\n\nSchema:\n{"action": "<remove_feature | simplify_feature | add_feature | stop>",\n "args": {...},\n "reason": "<one short sentence>"}\n\n# WORKED EXAMPLE — DO NOT COPY VERBATIM, JUST IMITATE THE SHAPE\nInput observation: complexity 11 exceeds limit 5; missing required types: ["safety"]\nInput candidates (already validated):\n  - {"tool": "add_feature", "args": {"name": "First-click safe", "complexity": 2, "feature_type": "safety", "required": true}, "score": 0.86, ...}\n  - {"tool": "remove_feature", "args": {"name": "Auto-flag completed"}, "score": 0.75, ...}\nYour reply (one line, no fences):\n{"action": "remove_feature", "args": {"name": "Auto-flag completed"}, "reason": "Drop the largest non-required feature first to make room for the missing safety feature."}\n\n# SAFE STRATEGY WHEN YOU ARE UNSURE\nThe "candidate actions" list in every step already contains valid tool calls\nwith valid args. The safest thing you can do is COPY one candidate\'s "tool"\nand "args" verbatim into your reply, then write your own one-sentence reason.\n\n# WHEN TO STOP\nIf the observation says `"passed": true`, or both constraints have been\nsatisfied, reply with:\n{"action": "stop", "args": {}, "reason": "all constraints pass"}\n'
MINESWEEPER_PLAY_SYSTEM_PROMPT = 'You are an Observable Minesweeper-Playing Agent for STEM teachers.\n\nYou are playing a small Minesweeper board. Each turn pick ONE tool call that\nbrings the game closer to clearing every non-mine cell, then output it as\nONE JSON object on a single line. Nothing else.\n\n# THE GAME\n- Board cells are identified by row letter + column number, e.g. A3, B2, F6.\n- Revealed numbered cell `K` means: exactly K of its 8 neighbours are mines.\n- `_` means hidden. `F` means you have flagged that cell as a suspected mine.\n- You WIN when every non-mine cell is revealed.\n- You LOSE immediately if you reveal a mine.\n\n# YOUR TOOLS (only the ones in state.allowed_actions; check it every turn)\n- reveal   args: {"row": <"A"-letter or 0-index int>, "col": <1-based int or 0-based int>}\n- flag     args: {"row": ..., "col": ...}\n- unflag   args: {"row": ..., "col": ...}\n- stop     args: {}\n\nThe row arg can be a letter ("A", "B", ...) OR an integer (0, 1, ...). The col\narg can be 1-based ("1"..."N") OR 0-based (0..N-1). Both are accepted by the\ngame engine. Pick whichever matches the candidates list verbatim.\n\n# WHAT YOU RECEIVE EACH TURN\n1. ASCII board (your view so far): `_`=hidden, `F`=flag, digits=neighbour mine count, `.`=zero mines around (auto-opened).\n2. JSON state with turn, status, score, unrevealed_count, flagged_count, allowed_actions.\n3. A short list of candidate moves derived from logical deduction.\n\n# OUTPUT FORMAT (rigid)\nReply with ONE JSON object on ONE line. No prose. No markdown. No code fences.\n\nSchema:\n{"action": "<one of allowed_actions>",\n "args": {"row": ..., "col": ...},\n "reason": "<one short sentence the student can read>"}\n\n# DEDUCTION HINTS (use these to write your reason)\n- If a revealed `K` has exactly K hidden neighbours → those are all mines → flag them.\n- If a revealed `K` already has K flagged neighbours → other hidden neighbours are safe → reveal them.\n- If no logical certainty exists → revealing a corner is usually safer than the middle.\n\n# SAFE STRATEGY WHEN YOU ARE UNSURE\nThe candidate list already contains valid moves with valid coordinates.\nIf you are unsure, COPY one candidate\'s "tool" and "args" verbatim, then\nwrite your own one-sentence reason. Never invent coordinates outside the\nboard.\n\n# WORKED EXAMPLE\nBoard:                State JSON:\n     1   2   3        {"turn": 3, "status": "playing", "score": 25,\n  A  _   1   _         "unrevealed_count": 5, "flagged_count": 0,\n  B  1   2   _         "allowed_actions": ["reveal", "flag", "unflag"]}\n  C  _   _   _\nCandidates:\n  - {"tool": "flag", "args": {"row": "A", "col": 3}, "score": 0.95,\n     "reason": "A2 shows 1 with A3 as only hidden neighbour — must be mine."}\nYour reply:\n{"action": "flag", "args": {"row": "A", "col": 3}, "reason": "A2 shows 1 and A3 is its only hidden neighbour — must be a mine."}\n\n# WHEN TO STOP\nIf the game has already ended or there are no legal moves left, reply with:\n{"action": "stop", "args": {}, "reason": "game already over"}\n'

def _format_observation_for_llm(observation: dict[str, Any]) -> str:
    complexity = observation['complexity']
    requirements = observation['requirements']
    return json.dumps({'passed': observation['passed'], 'complexity': {'total': complexity['total'], 'limit': complexity['limit'], 'passed': complexity['passed']}, 'missing_required_types': requirements.get('missing', []), 'issues': observation['issues']}, ensure_ascii=False)

def _format_candidates_for_llm(candidates: list[dict[str, Any]]) -> str:
    return json.dumps([{'label': item.get('label', item['tool']), 'tool': item['tool'], 'args': item['args'], 'score': item['score'], 'reason': item['reason']} for item in candidates], ensure_ascii=False)

def _format_spec_for_llm(spec: dict[str, Any]) -> str:
    return json.dumps({'title': spec.get('title'), 'complexity_limit': spec['complexity_limit'], 'must_include': spec['must_include'], 'features': [{'name': item['name'], 'complexity': item['complexity'], 'type': item['type'], 'required': item['required'], 'status': item['status']} for item in spec['features']]}, ensure_ascii=False)

def _extract_json(text: str) -> dict[str, Any] | None:
    """Pull the first JSON object out of an LLM reply (robust to extra prose)."""
    if not text:
        return None
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fenced = re.search('```(?:json)?\\s*(\\{.*?\\})\\s*```', text, re.DOTALL)
    if fenced:
        try:
            return json.loads(fenced.group(1))
        except json.JSONDecodeError:
            pass
    bare = re.search('\\{.*\\}', text, re.DOTALL)
    if bare:
        try:
            return json.loads(bare.group(0))
        except json.JSONDecodeError:
            return None
    return None

def llm_agent_loop(initial_spec: dict[str, Any], goal: str, observe_fn: Callable[[dict[str, Any]], dict[str, Any]], propose_fn: Callable[[dict[str, Any], dict[str, Any]], list[dict[str, Any]]], tools: dict[str, Callable[..., tuple[dict[str, Any], dict[str, Any]]]], *, max_steps: int=5, endpoint: str=DEFAULT_LM_STUDIO_ENDPOINT, model: str=DEFAULT_MODEL, on_step: Callable[[dict[str, Any]], None] | None=None, temperature: float=0.1) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    """Run a real LM Studio Agent loop over the game spec.

    At every iteration we:

    1. ``observe`` the current spec with Python tools.
    2. Hand the observation + candidate actions to LM Studio.
    3. Parse the LLM reply as one JSON tool call.
    4. ``act`` by running the chosen Python tool.
    5. ``observe`` again to reflect on the new spec.

    The loop stops when constraints pass, the LLM says ``stop``, or
    ``max_steps`` is reached. If LM Studio briefly misbehaves we fall back
    to the deterministic best candidate so you still end with a visible
    result.

    Pass ``on_step`` to stream each trace into the notebook UI in real time.
    """
    current_spec = initial_spec
    traces: list[dict[str, Any]] = []
    chat_history: list[dict[str, str]] = [{'role': 'system', 'content': AGENT_SYSTEM_PROMPT}]
    for step in range(1, max_steps + 1):
        observation = observe_fn(current_spec)
        candidates = propose_fn(current_spec, observation)
        user_message = f'Step {step}.\nGoal: {goal}\nSpec:\n{_format_spec_for_llm(current_spec)}\nObservation:\n{_format_observation_for_llm(observation)}\nCandidate actions (already validated):\n{_format_candidates_for_llm(candidates)}\nReply with ONE JSON tool call as instructed.'
        chat_history.append({'role': 'user', 'content': user_message})
        trace: dict[str, Any] = {'step': step, 'observation': observation, 'candidates': candidates, 'llm_raw': None, 'action': None, 'result': None, 'reflection': observation, 'source': 'llm'}
        if observation['passed']:
            trace['action'] = {'tool': 'stop', 'args': {}, 'reason': 'all constraints pass'}
            trace['result'] = {'success': True, 'message': 'no action needed'}
            traces.append(trace)
            if on_step is not None:
                on_step(trace)
            break
        try:
            reply = call_local_lmstudio(chat_history, endpoint=endpoint, model=model, temperature=temperature)
            trace['llm_raw'] = reply
            chat_history.append({'role': 'assistant', 'content': reply})
            decision = _extract_json(reply)
        except Exception as exc:
            trace['llm_raw'] = f'[LM Studio unavailable: {exc}]'
            decision = None
        if not decision or 'action' not in decision:
            trace['source'] = 'fallback'
            if not candidates:
                trace['action'] = None
                trace['result'] = {'success': False, 'message': 'no candidate available'}
                traces.append(trace)
                if on_step is not None:
                    on_step(trace)
                break
            best = max(candidates, key=lambda item: item['score'])
            decision = {'action': best['tool'], 'args': best['args'], 'reason': f'fallback to deterministic best candidate: {best['reason']}'}
        action_name = decision.get('action')
        if action_name == 'stop':
            trace['action'] = {'tool': 'stop', 'args': {}, 'reason': decision.get('reason', '')}
            trace['result'] = {'success': True, 'message': 'LLM stopped the loop'}
            traces.append(trace)
            if on_step is not None:
                on_step(trace)
            break
        tool = tools.get(action_name)
        if tool is None:
            trace['action'] = {'tool': action_name, 'args': decision.get('args', {}), 'reason': decision.get('reason', '')}
            trace['result'] = {'success': False, 'message': f'unknown tool: {action_name}'}
            traces.append(trace)
            if on_step is not None:
                on_step(trace)
            break
        args = decision.get('args') or {}
        try:
            new_spec, result = tool(current_spec, **args)
        except TypeError as exc:
            trace['action'] = {'tool': action_name, 'args': args, 'reason': decision.get('reason', '')}
            trace['result'] = {'success': False, 'message': f'bad args for {action_name}: {exc}'}
            traces.append(trace)
            if on_step is not None:
                on_step(trace)
            break
        reflection = observe_fn(new_spec)
        trace['action'] = {'tool': action_name, 'args': args, 'reason': decision.get('reason', '')}
        trace['result'] = result
        trace['reflection'] = reflection
        traces.append(trace)
        if on_step is not None:
            on_step(trace)
        current_spec = new_spec
        chat_history.append({'role': 'user', 'content': f'Tool result: {json.dumps(result, ensure_ascii=False)}\nReflection: {_format_observation_for_llm(reflection)}'})
        if reflection['passed']:
            break
    return (current_spec, traces)
'Minesweeper game engine + CLI used by Case 4 Phase B.\n\nThe agent reads the game state with ``observe_for_llm`` and picks a single\nmove per turn (``reveal``/``flag``/``unflag``) constrained by what the\nfinalised spec allows. The notebook drives this loop one move at a time\nthrough the ipywidgets shell in ``utils.minesweeper_view``.\n\nRun directly to play in a terminal::\n\n    python -m utils.minesweeper_game --size 6 --mines 6\n'
import argparse
import random
import string
from copy import deepcopy
from typing import Any
DEFAULT_BOARD_SIZE = 6
DEFAULT_MINE_COUNT = 6
MAX_BOARD_SIZE = len(string.ascii_uppercase)
ActionResult = dict[str, Any]
Coord = tuple[int, int]

def parse_coord(raw: Any, board_size: int) -> Coord:
    """Normalise a coordinate to a zero-indexed (row, col) tuple.

    Accepted shapes:
        "A3" / "a3"                     -> (0, 2)
        {"row": "B", "col": 4}          -> (1, 3)
        {"row": 1, "col": 3}            -> (1, 3)
        ("B", 4)                        -> (1, 3)
        [1, 3]                          -> (1, 3)
    """
    if isinstance(raw, dict):
        row_raw, col_raw = (raw.get('row'), raw.get('col'))
    elif isinstance(raw, (list, tuple)) and len(raw) == 2:
        row_raw, col_raw = raw
    elif isinstance(raw, str):
        text = raw.strip().upper()
        if not text or not text[0].isalpha():
            raise ValueError(f'cannot parse coord {raw!r}')
        row_raw, col_raw = (text[0], text[1:])
    else:
        raise ValueError(f'cannot parse coord {raw!r}')
    if isinstance(row_raw, str):
        row_raw = row_raw.strip().upper()
        if len(row_raw) != 1 or not row_raw.isalpha():
            raise ValueError(f'row must be a single letter, got {row_raw!r}')
        row = ord(row_raw) - ord('A')
    else:
        row = int(row_raw)
    if isinstance(col_raw, str):
        col_text = col_raw.strip()
        if not col_text.isdigit():
            raise ValueError(f'col must be an integer, got {col_raw!r}')
        col = int(col_text) - 1
    else:
        col = int(col_raw) - 1
    if not (0 <= row < board_size and 0 <= col < board_size):
        raise ValueError(f'coord ({row}, {col}) out of bounds for {board_size}x{board_size} board')
    return (row, col)

def format_coord(row: int, col: int) -> str:
    return f'{string.ascii_uppercase[row]}{col + 1}'

class MinesweeperGame:
    """Minimal Minesweeper engine. All mutating methods return a ToolResult."""

    def __init__(self, board_size: int=DEFAULT_BOARD_SIZE, mine_count: int=DEFAULT_MINE_COUNT, first_click_safe: bool=True, flood_fill: bool=True, seed: int | None=None):
        if not 2 <= board_size <= MAX_BOARD_SIZE:
            raise ValueError(f'board_size must be in [2, {MAX_BOARD_SIZE}]')
        if not 1 <= mine_count < board_size * board_size:
            raise ValueError(f'mine_count must be in [1, {board_size ** 2 - 1}]')
        self.board_size = board_size
        self.mine_count = mine_count
        self.first_click_safe = first_click_safe
        self.flood_fill = flood_fill
        self._rng = random.Random(seed)
        self.mines: list[list[bool]] = [[False] * board_size for _ in range(board_size)]
        self.revealed: list[list[bool]] = [[False] * board_size for _ in range(board_size)]
        self.flagged: list[list[bool]] = [[False] * board_size for _ in range(board_size)]
        self.status: str = 'playing'
        self.turn: int = 0
        self.first_click_done: bool = False
        self.history: list[dict] = []
        self.exploded_at: Coord | None = None
        self._place_mines()

    def _place_mines(self, forbidden: set[Coord] | None=None) -> None:
        forbidden = forbidden or set()
        cells = [(r, c) for r in range(self.board_size) for c in range(self.board_size) if (r, c) not in forbidden]
        chosen = self._rng.sample(cells, self.mine_count)
        for r, c in chosen:
            self.mines[r][c] = True

    def _replace_mine_after_first_click(self, safe: Coord) -> None:
        r0, c0 = safe
        if not self.mines[r0][c0]:
            return
        self.mines[r0][c0] = False
        for r in range(self.board_size):
            for c in range(self.board_size):
                if not self.mines[r][c] and (r, c) != safe:
                    self.mines[r][c] = True
                    return

    def _neighbors(self, row: int, col: int) -> list[Coord]:
        out = []
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == 0 and dc == 0:
                    continue
                nr, nc = (row + dr, col + dc)
                if 0 <= nr < self.board_size and 0 <= nc < self.board_size:
                    out.append((nr, nc))
        return out

    def neighbor_mine_count(self, row: int, col: int) -> int:
        return sum((1 for r, c in self._neighbors(row, col) if self.mines[r][c]))

    def reveal(self, row: int, col: int) -> ActionResult:
        if self.status != 'playing':
            return {'success': False, 'message': f'game already {self.status}'}
        if self.flagged[row][col]:
            return {'success': False, 'message': f'{format_coord(row, col)} is flagged; unflag first'}
        if self.revealed[row][col]:
            return {'success': False, 'message': f'{format_coord(row, col)} already revealed'}
        if self.first_click_safe and (not self.first_click_done):
            self._replace_mine_after_first_click((row, col))
        self.first_click_done = True
        self.turn += 1
        if self.mines[row][col]:
            self.revealed[row][col] = True
            self.exploded_at = (row, col)
            self.status = 'lost'
            self._record({'action': 'reveal', 'args': {'row': row, 'col': col}}, {'success': True, 'message': f'BOOM at {format_coord(row, col)}', 'outcome': 'lost'})
            return {'success': True, 'message': f'BOOM at {format_coord(row, col)}', 'outcome': 'lost'}
        opened = self._flood_reveal(row, col)
        outcome = 'won' if self._check_won() else 'playing'
        if outcome == 'won':
            self.status = 'won'
        msg = f'revealed {len(opened)} cell(s) starting at {format_coord(row, col)}'
        self._record({'action': 'reveal', 'args': {'row': row, 'col': col}}, {'success': True, 'message': msg, 'outcome': outcome, 'opened': opened})
        return {'success': True, 'message': msg, 'outcome': outcome, 'opened': opened}

    def _flood_reveal(self, row: int, col: int) -> list[Coord]:
        opened: list[Coord] = []
        stack = [(row, col)]
        while stack:
            r, c = stack.pop()
            if self.revealed[r][c] or self.flagged[r][c] or self.mines[r][c]:
                continue
            self.revealed[r][c] = True
            opened.append((r, c))
            if self.flood_fill and self.neighbor_mine_count(r, c) == 0:
                stack.extend(self._neighbors(r, c))
        return opened

    def flag(self, row: int, col: int) -> ActionResult:
        if self.status != 'playing':
            return {'success': False, 'message': f'game already {self.status}'}
        if self.revealed[row][col]:
            return {'success': False, 'message': f'{format_coord(row, col)} already revealed; cannot flag'}
        if self.flagged[row][col]:
            return {'success': False, 'message': f'{format_coord(row, col)} already flagged'}
        self.flagged[row][col] = True
        self.turn += 1
        self._record({'action': 'flag', 'args': {'row': row, 'col': col}}, {'success': True, 'message': f'flagged {format_coord(row, col)}'})
        return {'success': True, 'message': f'flagged {format_coord(row, col)}'}

    def unflag(self, row: int, col: int) -> ActionResult:
        if self.status != 'playing':
            return {'success': False, 'message': f'game already {self.status}'}
        if not self.flagged[row][col]:
            return {'success': False, 'message': f'{format_coord(row, col)} is not flagged'}
        self.flagged[row][col] = False
        self.turn += 1
        self._record({'action': 'unflag', 'args': {'row': row, 'col': col}}, {'success': True, 'message': f'unflagged {format_coord(row, col)}'})
        return {'success': True, 'message': f'unflagged {format_coord(row, col)}'}

    def _record(self, decision: dict, result: dict) -> None:
        self.history.append({'turn': self.turn, 'decision': decision, 'result': result})

    def _check_won(self) -> bool:
        for r in range(self.board_size):
            for c in range(self.board_size):
                if not self.mines[r][c] and (not self.revealed[r][c]):
                    return False
        return True

def apply_decision(game: MinesweeperGame, decision: dict) -> tuple[MinesweeperGame, ActionResult]:
    """Apply one decision JSON to the game. Returns (game, result).

    The game object is mutated in place; the same object is returned so the
    caller can chain. Tolerates loose coordinate formats.
    """
    action = decision.get('action')
    args = decision.get('args') or {}
    if action == 'stop':
        return (game, {'success': True, 'message': 'agent chose to stop', 'outcome': game.status})
    if action not in ('reveal', 'flag', 'unflag'):
        return (game, {'success': False, 'message': f'unknown action: {action!r}'})
    try:
        if 'row' in args and 'col' in args:
            row, col = parse_coord({'row': args['row'], 'col': args['col']}, game.board_size)
        elif 'cell' in args:
            row, col = parse_coord(args['cell'], game.board_size)
        else:
            return (game, {'success': False, 'message': f'missing row/col in args: {args!r}'})
    except (ValueError, TypeError) as exc:
        return (game, {'success': False, 'message': f'bad coordinate: {exc}'})
    method = getattr(game, action)
    return (game, method(row, col))

def _cell_glyph(game: MinesweeperGame, row: int, col: int, reveal_mines: bool=False) -> str:
    if game.revealed[row][col]:
        if game.mines[row][col]:
            return '*'
        n = game.neighbor_mine_count(row, col)
        return str(n) if n else '.'
    if reveal_mines and game.mines[row][col]:
        return '*'
    if game.flagged[row][col]:
        return 'F'
    return '_'

def render_ascii(game: MinesweeperGame, reveal_mines: bool=False) -> str:
    n = game.board_size
    header = '     ' + '  '.join((f'{c + 1:>2}' for c in range(n)))
    lines = [header]
    for r in range(n):
        row_letter = string.ascii_uppercase[r]
        cells = '  '.join((f'{_cell_glyph(game, r, c, reveal_mines):>2}' for c in range(n)))
        lines.append(f'  {row_letter}  {cells}')
    return '\n'.join(lines)

def render_html(game: MinesweeperGame, reveal_mines: bool | None=None) -> str:
    """Return an HTML table suitable for ipywidgets.HTML.

    ``reveal_mines`` defaults to True when the game is over (loss).
    """
    if reveal_mines is None:
        reveal_mines = game.status == 'lost'
    n = game.board_size
    css = '\n    <style>\n      .ms-board { border-collapse: collapse; margin: 8px auto; font-family: monospace; }\n      .ms-board th, .ms-board td {\n          width: 36px; height: 36px; text-align: center; font-size: 18px;\n          border: 1px solid #888; padding: 0;\n      }\n      .ms-board th { background:#eaeaea; color:#444; font-weight: 600; }\n      .ms-hidden { background:#cfd8dc; }\n      .ms-flag   { background:#ffe082; color:#bf360c; font-weight:bold; }\n      .ms-open   { background:#fafafa; color:#1a237e; font-weight:bold; }\n      .ms-zero   { background:#fafafa; color:#9e9e9e; }\n      .ms-mine   { background:#ef9a9a; color:#b71c1c; font-weight:bold; }\n      .ms-boom   { background:#d32f2f; color:#fff; font-weight:bold; }\n    </style>\n    '
    rows = ['<tr><th></th>' + ''.join((f'<th>{c + 1}</th>' for c in range(n))) + '</tr>']
    for r in range(n):
        cells = [f'<th>{string.ascii_uppercase[r]}</th>']
        for c in range(n):
            if game.exploded_at == (r, c):
                cells.append('<td class="ms-boom">*</td>')
                continue
            if game.revealed[r][c]:
                if game.mines[r][c]:
                    cells.append('<td class="ms-mine">*</td>')
                else:
                    val = game.neighbor_mine_count(r, c)
                    if val == 0:
                        cells.append('<td class="ms-zero">·</td>')
                    else:
                        cells.append(f'<td class="ms-open">{val}</td>')
            elif reveal_mines and game.mines[r][c]:
                cells.append('<td class="ms-mine">*</td>')
            elif game.flagged[r][c]:
                cells.append('<td class="ms-flag">F</td>')
            else:
                cells.append('<td class="ms-hidden">·</td>')
        rows.append('<tr>' + ''.join(cells) + '</tr>')
    table = '<table class="ms-board">' + ''.join(rows) + '</table>'
    return css + table

def compute_score(game: MinesweeperGame) -> dict[str, Any]:
    n = game.board_size
    safe_cells_total = n * n - game.mine_count
    revealed_safe = sum((1 for r in range(n) for c in range(n) if game.revealed[r][c] and (not game.mines[r][c])))
    correct_flags = sum((1 for r in range(n) for c in range(n) if game.flagged[r][c] and game.mines[r][c]))
    wrong_flags = sum((1 for r in range(n) for c in range(n) if game.flagged[r][c] and (not game.mines[r][c])))
    win_bonus = 50 if game.status == 'won' else 0
    score = revealed_safe * 10 + correct_flags * 5 - wrong_flags * 5 + win_bonus
    max_score = safe_cells_total * 10 + game.mine_count * 5 + 50
    return {'score': score, 'max_score': max_score, 'revealed_safe': revealed_safe, 'safe_cells_total': safe_cells_total, 'correct_flags': correct_flags, 'wrong_flags': wrong_flags, 'win_bonus': win_bonus, 'status': game.status, 'turn': game.turn}

def observe_for_llm(game: MinesweeperGame, spec: dict | None=None) -> dict[str, Any]:
    allowed_actions = (spec or {}).get('allowed_actions', ['reveal', 'flag', 'unflag'])
    score_info = compute_score(game)
    unrevealed = sum((1 for r in range(game.board_size) for c in range(game.board_size) if not game.revealed[r][c] and (not game.flagged[r][c])))
    flagged_total = sum((1 for r in range(game.board_size) for c in range(game.board_size) if game.flagged[r][c]))
    last_entry = game.history[-1] if game.history else None
    return {'board_ascii': render_ascii(game), 'state': {'turn': game.turn, 'status': game.status, 'score': score_info['score'], 'unrevealed_count': unrevealed, 'flagged_count': flagged_total, 'mines_total': game.mine_count, 'allowed_actions': list(allowed_actions), 'board_size': game.board_size}, 'last_action': last_entry}

def propose_minesweeper_actions(game: MinesweeperGame, spec: dict | None=None) -> list[dict[str, Any]]:
    """Generate ranked candidates so a weak LLM can copy one verbatim.

    Algorithm:
        - For every revealed numbered cell, do classic minesweeper deduction
          to derive certain flags and certain reveals.
        - If no certainty exists, propose a low-risk first-move heuristic
          (corners > edges > centre).
        - Filter the final list by ``spec["allowed_actions"]`` so the
          candidates always respect what the spec permits.
    """
    if game.status != 'playing':
        return []
    spec = spec or {}
    allowed = set(spec.get('allowed_actions', ['reveal', 'flag', 'unflag']))
    candidates: list[dict[str, Any]] = []
    seen: set[tuple[str, int, int]] = set()

    def add(action: str, row: int, col: int, score: float, reason: str, label: str | None=None) -> None:
        if action not in allowed:
            return
        key = (action, row, col)
        if key in seen:
            return
        seen.add(key)
        candidates.append({'label': label or f'{action} {format_coord(row, col)}', 'tool': action, 'args': {'row': format_coord(row, col)[0], 'col': col + 1}, 'score': score, 'reason': reason})
    n = game.board_size
    for r in range(n):
        for c in range(n):
            if not game.revealed[r][c] or game.mines[r][c]:
                continue
            value = game.neighbor_mine_count(r, c)
            if value == 0:
                continue
            unrevealed_neighbors = [(nr, nc) for nr, nc in game._neighbors(r, c) if not game.revealed[nr][nc] and (not game.flagged[nr][nc])]
            flagged_neighbors = [(nr, nc) for nr, nc in game._neighbors(r, c) if game.flagged[nr][nc]]
            if unrevealed_neighbors and value - len(flagged_neighbors) == len(unrevealed_neighbors):
                for nr, nc in unrevealed_neighbors:
                    add('flag', nr, nc, 0.95, f'{format_coord(r, c)} shows {value} and has exactly {len(unrevealed_neighbors)} hidden neighbor(s) — {format_coord(nr, nc)} must be a mine.')
            if unrevealed_neighbors and len(flagged_neighbors) == value:
                for nr, nc in unrevealed_neighbors:
                    add('reveal', nr, nc, 0.9, f'{format_coord(r, c)} shows {value} with {value} flagged neighbor(s) — {format_coord(nr, nc)} is safe.')
    if not candidates:
        ranking: list[tuple[int, Coord]] = []
        for r in range(n):
            for c in range(n):
                if game.revealed[r][c] or game.flagged[r][c]:
                    continue
                corner = r in (0, n - 1) and c in (0, n - 1)
                edge = r in (0, n - 1) or c in (0, n - 1)
                priority = 0 if corner else 1 if edge else 2
                ranking.append((priority, (r, c)))
        ranking.sort(key=lambda t: t[0])
        for priority, (r, c) in ranking[:3]:
            tag = 'corner' if priority == 0 else 'edge' if priority == 1 else 'centre'
            add('reveal', r, c, 0.55 - 0.05 * priority, f'No certain move yet — guessing {tag} {format_coord(r, c)} which is statistically safer.')
    candidates.sort(key=lambda item: item['score'], reverse=True)
    return candidates[:6]

def _cli_help() -> str:
    return 'Commands:\n  r<row><col>   reveal a cell, e.g. rA3\n  f<row><col>   flag a cell,   e.g. fB2\n  u<row><col>   unflag a cell, e.g. uB2\n  h             show this help\n  q             quit'

def _cli_parse(raw: str, board_size: int) -> tuple[str, Coord] | None:
    raw = raw.strip().lower()
    if not raw:
        return None
    cmd, rest = (raw[0], raw[1:])
    if cmd not in ('r', 'f', 'u'):
        return None
    try:
        row, col = parse_coord(rest, board_size)
    except ValueError:
        return None
    return ({'r': 'reveal', 'f': 'flag', 'u': 'unflag'}[cmd], (row, col))

def play_in_terminal(board_size: int, mine_count: int, seed: int | None=None) -> int:
    game = MinesweeperGame(board_size=board_size, mine_count=mine_count, seed=seed)
    print(f'Minesweeper {board_size}x{board_size} with {mine_count} mines.')
    print(_cli_help())
    while game.status == 'playing':
        print()
        print(render_ascii(game))
        score = compute_score(game)
        print(f'Score: {score['score']} / {score['max_score']}   Turn: {game.turn}')
        try:
            raw = input('> ').strip().lower()
        except (EOFError, KeyboardInterrupt):
            print()
            return 130
        if raw in ('q', 'quit', 'exit'):
            print('bye')
            return 0
        if raw in ('h', 'help', '?'):
            print(_cli_help())
            continue
        parsed = _cli_parse(raw, board_size)
        if parsed is None:
            print('unrecognised command. type h for help.')
            continue
        action, (row, col) = parsed
        result = getattr(game, action)(row, col)
        print(result.get('message', ''))
    print()
    print(render_ascii(game, reveal_mines=True))
    score = compute_score(game)
    outcome = 'WIN' if game.status == 'won' else 'LOSE'
    print(f'\n{outcome}   Final score: {score['score']} / {score['max_score']}')
    print(f'   revealed_safe={score['revealed_safe']}/{score['safe_cells_total']}   correct_flags={score['correct_flags']}/{game.mine_count}   wrong_flags={score['wrong_flags']}   turn={score['turn']}')
    return 0 if game.status == 'won' else 1

def main() -> int:
    p = argparse.ArgumentParser(prog='utils.minesweeper_game')
    p.add_argument('--size', type=int, default=DEFAULT_BOARD_SIZE)
    p.add_argument('--mines', type=int, default=DEFAULT_MINE_COUNT)
    p.add_argument('--seed', type=int, default=None)
    args = p.parse_args()
    return play_in_terminal(args.size, args.mines, args.seed)
'Fully autonomous Minesweeper agent loop (no candidate list).\n\nThe constrained widget loop in ``minesweeper_view`` hands the LLM a\npre-validated candidate list and can fall back to the best one. This module\ndoes the opposite: the model must produce a move itself. That freedom needs\nthree guardrails a candidate-free agent cannot skip:\n\n1. **Action history** in the prompt, so the model stops re-picking a cell.\n2. **Retry with the error fed back**, when the reply is not parseable JSON or\n   the coordinate is off the board.\n3. Only a **structurally valid, in-range** decision is applied.\n\nThe loop is injectable (``call_fn``) so tests drive it with a fake LLM and no\nnetwork. Network/LLM errors surface inside the trace instead of crashing, so a\nclassroom demo always finishes.\n'
import json
from typing import Any, Callable
MINESWEEPER_FREE_SYSTEM_PROMPT = 'You are an expert Minesweeper Agent playing a small board on your own.\nThere is NO candidate list this turn — you must choose the move yourself.\n\n# THE GAME\n- Cells are named by a row letter and a column number, e.g. A1, B3, C4.\n- A revealed number K means exactly K of that cell\'s 8 neighbours are mines.\n- `_` is hidden, `F` is a cell you flagged, `.` is an auto-opened zero.\n- WIN by revealing every non-mine cell. LOSE the instant you reveal a mine.\n\n# TOOLS\n- reveal / flag / unflag, args {"row": "<A..>", "col": <1..N>}\n- stop, args {}\n\n# OUTPUT (rigid)\nReply with ONE JSON object on ONE line, no prose, no code fences:\n{"action": "reveal|flag|unflag|stop", "args": {"row": "A", "col": 1}, "reason": "<one short sentence>"}\n\n# RULES YOU MUST FOLLOW\n- Only use row letters and column numbers that exist on the board.\n- Never pick a cell that already appears in the "already tried" list below.\n- If a number K has exactly K hidden neighbours, they are all mines -> flag one.\n- If a number K already has K flags around it, its other hidden neighbours are safe -> reveal one.\n- With no certainty, a corner is a safer guess than the centre.\n'

def _decision_cell(decision: dict[str, Any]) -> str:
    args = decision.get('args') or {}
    return f'{args.get('row')}{args.get('col')}'

def _history_lines(action_log: list[dict[str, Any]]) -> str:
    if not action_log:
        return '(none yet)'
    lines = []
    for entry in action_log:
        decision = entry['decision']
        outcome = 'ok' if entry['result'].get('success') else 'FAILED'
        lines.append(f'{decision.get('action')} {_decision_cell(decision)} -> {outcome}: {entry['result'].get('message', '')}')
    return '\n'.join(lines)

def build_free_messages(game: MinesweeperGame, action_log: list[dict[str, Any]] | None=None, error_feedback: str | None=None) -> list[dict[str, str]]:
    """Build the candidate-free chat messages for one turn."""
    observation = observe_for_llm(game)
    user = f'Board:\n{observation['board_ascii']}\n\nState: {json.dumps(observation['state'], ensure_ascii=False)}\n\nAlready tried (do not repeat a failed cell):\n{_history_lines(action_log or [])}\n\nChoose ONE move and reply with ONE JSON object.'
    if error_feedback:
        user += f'\n\nYour previous reply was rejected: {error_feedback}\nTry again with valid JSON and an on-board coordinate.'
    return [{'role': 'system', 'content': MINESWEEPER_FREE_SYSTEM_PROMPT}, {'role': 'user', 'content': user}]

def validate_decision(decision: dict[str, Any] | None, board_size: int) -> tuple[bool, str]:
    """Structural + range check (not semantic). Returns (ok, error_message).

    A ``True`` result only guarantees the shape and that the coordinate sits on
    the board. Whether the move is *useful* (e.g. the cell is already revealed)
    is decided later by the game engine.
    """
    if not isinstance(decision, dict) or 'action' not in decision:
        return (False, "no JSON object with an 'action' field")
    action = decision.get('action')
    if action == 'stop':
        return (True, '')
    if action not in ('reveal', 'flag', 'unflag'):
        return (False, f'unknown action {action!r}; use reveal, flag, unflag, or stop')
    args = decision.get('args') or {}
    if 'row' not in args or 'col' not in args:
        return (False, "args must contain 'row' and 'col'")
    try:
        parse_coord({'row': args['row'], 'col': args['col']}, board_size)
    except (ValueError, TypeError) as exc:
        return (False, f'coordinate off the board: {exc}')
    return (True, '')

def ask_free_move(game: MinesweeperGame, *, model: str=DEFAULT_MODEL, endpoint: str=DEFAULT_LM_STUDIO_ENDPOINT, temperature: float=0.2, call_fn: Callable[..., str]=call_local_lmstudio, action_log: list[dict[str, Any]] | None=None) -> tuple[str, dict[str, Any] | None]:
    """One candidate-free request. Returns (raw_reply, parsed_decision_or_None).

    Used by the "free generation vs candidates" experiment to show a small
    model's raw output before any guardrail cleans it up.
    """
    messages = build_free_messages(game, action_log=action_log)
    raw = call_fn(messages, endpoint=endpoint, model=model, temperature=temperature)
    return (raw, _extract_json(raw))

def autonomous_agent_loop(game: MinesweeperGame, *, model: str=DEFAULT_MODEL, endpoint: str=DEFAULT_LM_STUDIO_ENDPOINT, max_steps: int=8, max_retries: int=2, temperature: float=0.2, call_fn: Callable[..., str]=call_local_lmstudio, on_step: Callable[[dict[str, Any]], None] | None=None) -> list[dict[str, Any]]:
    """Play the game with NO candidate list. Returns a list of per-step traces.

    Each step: observe -> ask the model freely -> validate (retry with the
    error fed back on bad JSON or off-board coordinates) -> apply -> record the
    move in the action history so the next turn can avoid repeats. Stops on
    win, loss, an explicit ``stop``, exhausted retries, or ``max_steps``.
    """
    traces: list[dict[str, Any]] = []
    action_log: list[dict[str, Any]] = []
    for step in range(1, max_steps + 1):
        if game.status != 'playing':
            break
        observation = observe_for_llm(game)
        decision: dict[str, Any] | None = None
        raw: str | None = None
        error_feedback: str | None = None
        retries = 0
        for attempt in range(max_retries + 1):
            messages = build_free_messages(game, action_log=action_log, error_feedback=error_feedback)
            try:
                raw = call_fn(messages, endpoint=endpoint, model=model, temperature=temperature)
            except Exception as exc:
                raw = f'[LLM unavailable: {exc}]'
                error_feedback = str(exc)
                retries = attempt
                decision = None
                break
            parsed = _extract_json(raw)
            ok, err = validate_decision(parsed, game.board_size)
            if ok:
                decision = parsed
                retries = attempt
                break
            error_feedback = err
            retries = attempt + 1
            decision = None
        trace: dict[str, Any] = {'step': step, 'board_ascii': observation['board_ascii'], 'board_after': observation['board_ascii'], 'state': observation['state'], 'llm_raw': raw, 'retries': retries, 'decision': decision, 'result': None, 'status': game.status}
        if decision is None:
            trace['result'] = {'success': False, 'message': f'gave up after {max_retries} retr{('y' if max_retries == 1 else 'ies')}: {error_feedback}'}
            traces.append(trace)
            if on_step is not None:
                on_step(trace)
            break
        if decision.get('action') == 'stop':
            trace['result'] = {'success': True, 'message': 'agent chose to stop'}
            traces.append(trace)
            if on_step is not None:
                on_step(trace)
            break
        game, result = apply_decision(game, decision)
        trace['result'] = result
        trace['status'] = game.status
        trace['board_after'] = render_ascii(game)
        action_log.append({'decision': decision, 'result': result})
        traces.append(trace)
        if on_step is not None:
            on_step(trace)
        if game.status != 'playing':
            break
    return traces
'ipywidgets shell that lets students watch the Agent play Minesweeper\nAND let them click the board themselves.\n\nPublic entry point::\n\n    from utils.minesweeper_view import build_minesweeper_widget\n    ui = build_minesweeper_widget(spec=approved_spec, model_name=MODEL_NAME,\n                                  temperature=0.7)\n    ui\n\nUI layout:\n    1. score bar\n    2. clickable board (each cell is a button; click mode = reveal / flag / unflag)\n    3. mode radio (which action a cell click performs)\n    4. controls row: [Next Move (LLM)] [Auto-play 5] [Reset]\n    5. config row: board size + mines + temperature + [New Game]\n    6. trace area (LLM moves + your moves + fallback)\n'
import time
import traceback
from typing import Any, Callable
import ipywidgets as widgets
DEFAULT_TEMPERATURE = 0.7

def _new_game_from_spec(spec: dict) -> MinesweeperGame:
    return MinesweeperGame(board_size=spec.get('board_size', DEFAULT_BOARD_SIZE), mine_count=spec.get('mine_count', DEFAULT_MINE_COUNT), first_click_safe=spec.get('first_click_safe', True), flood_fill=spec.get('flood_fill', True), seed=spec.get('seed'))

def _render_score_bar(game: MinesweeperGame) -> str:
    score = compute_score(game)
    status_color = {'playing': '#1976d2', 'won': '#2e7d32', 'lost': '#c62828'}.get(game.status, '#333')
    status_label = {'playing': 'playing', 'won': 'WIN 🏆', 'lost': 'GAME OVER 💥'}.get(game.status, game.status)
    return f'<div style="font-family:monospace; padding:6px;"><b>Score:</b> {score['score']} / {score['max_score']} &nbsp;|&nbsp; <b>Turn:</b> {game.turn} &nbsp;|&nbsp; <b>Mines:</b> {game.mine_count} &nbsp;|&nbsp; <b>Flagged:</b> {score['correct_flags'] + score['wrong_flags']} &nbsp;|&nbsp; <span style="color:{status_color};font-weight:bold;">{status_label}</span></div>'

def _cell_descriptor(game: MinesweeperGame, r: int, c: int) -> tuple[str, str]:
    """Return (description, button_style) for one cell button."""
    if game.exploded_at == (r, c):
        return ('*', 'danger')
    if game.revealed[r][c]:
        if game.mines[r][c]:
            return ('*', 'danger')
        n = game.neighbor_mine_count(r, c)
        return (str(n) if n else '·', 'info' if n else '')
    if game.status == 'lost' and game.mines[r][c]:
        return ('*', 'danger')
    if game.flagged[r][c]:
        return ('F', 'warning')
    return (' ', '')

def _format_candidates_for_llm(candidates: list[dict]) -> str:
    import json as _json
    return _json.dumps([{'label': c.get('label', c['tool']), 'tool': c['tool'], 'args': c['args'], 'score': c['score'], 'reason': c['reason']} for c in candidates], ensure_ascii=False)

def _build_messages(observation: dict, candidates: list[dict], goal: str) -> list[dict]:
    import json as _json
    user = f'Goal: {goal}\nBoard (ASCII):\n{observation['board_ascii']}\n\nState JSON: {_json.dumps(observation['state'], ensure_ascii=False)}\n\nLast action: {_json.dumps(observation.get('last_action'), ensure_ascii=False)}\n\nCandidate moves (derived by Python deduction, copy verbatim if unsure):\n{_format_candidates_for_llm(candidates)}\n\nReply with ONE JSON tool call as instructed.'
    return [{'role': 'system', 'content': MINESWEEPER_PLAY_SYSTEM_PROMPT}, {'role': 'user', 'content': user}]

def _decide_one_move(game: MinesweeperGame, spec: dict, goal: str, model: str, endpoint: str, temperature: float=DEFAULT_TEMPERATURE) -> tuple[dict, dict, str, str | None]:
    """One Observe → Plan cycle. Returns (observation, decision, source, raw_reply).

    source ∈ {"llm", "fallback"}.
    """
    observation = observe_for_llm(game, spec)
    candidates = propose_minesweeper_actions(game, spec)
    if not candidates:
        return (observation, {'action': 'stop', 'args': {}, 'reason': 'no moves available'}, 'fallback', None)
    messages = _build_messages(observation, candidates, goal)
    raw_reply: str | None = None
    decision: dict | None = None
    try:
        raw_reply = call_local_lmstudio(messages, endpoint=endpoint, model=model, temperature=temperature)
        decision = _extract_json(raw_reply)
    except Exception as exc:
        raw_reply = f'[LLM unavailable: {exc}]'
    if not decision or 'action' not in decision:
        best = max(candidates, key=lambda c: c['score'])
        decision = {'action': best['tool'], 'args': best['args'], 'reason': f'fallback to deterministic best candidate: {best['reason']}'}
        return (observation, decision, 'fallback', raw_reply)
    return (observation, decision, 'llm', raw_reply)

def _append_trace_html(trace_widget: widgets.Output, turn: int, decision: dict, result: dict, source: str, raw_reply: str | None) -> None:
    palette = {'llm': ('#2e7d32', '🤖', 'LLM'), 'fallback': ('#ef6c00', '🛟', 'fallback'), 'human': ('#1565c0', '👤', 'you')}
    color, icon, label = palette.get(source, ('#555', '•', source))
    args = decision.get('args', {})
    lines = [f'<div style="border-left:3px solid {color}; padding:6px 10px; margin:6px 0; font-family:monospace; background:#f7f7f7;">', f'<b>Turn {turn}</b> &nbsp; <span style="color:{color};">[{label}]</span> &nbsp; {icon} <b>{decision.get('action')}</b> at <b>{args.get('row', '?')}{args.get('col', '?')}</b>', f'<br>&nbsp;&nbsp;💭 <i>{decision.get('reason', '')}</i>', f'<br>&nbsp;&nbsp;📌 {result.get('message', '')}']
    if raw_reply and source == 'llm':
        lines.append(f'<br>&nbsp;&nbsp;<details><summary style="color:#666; cursor:pointer;">show LLM raw reply</summary><pre style="white-space:pre-wrap; font-size:12px; color:#555;">{raw_reply[:600]}</pre></details>')
    lines.append('</div>')
    with trace_widget:
        from IPython.display import HTML, display
        display(HTML(''.join(lines)))

def build_minesweeper_widget(spec: dict, model_name: str, endpoint: str=DEFAULT_LM_STUDIO_ENDPOINT, goal: str | None=None, decide_fn: Callable | None=None, auto_play_pause: float=0.6, temperature: float=DEFAULT_TEMPERATURE) -> widgets.VBox:
    """Build the live Minesweeper UI.

    Includes click-to-act board, mode toggle (reveal / flag / unflag), LLM
    next-move button, auto-play, reset, and live config inputs for board
    size, mine count, and LLM temperature.

    ``decide_fn`` is an injection seam used by tests; production callers
    leave it as ``None`` to talk to LM Studio.
    """

    def _make_goal(spec_dict: dict) -> str:
        return f'Clear all non-mine cells on a {spec_dict.get('board_size', DEFAULT_BOARD_SIZE)}x{spec_dict.get('board_size', DEFAULT_BOARD_SIZE)} board with {spec_dict.get('mine_count', DEFAULT_MINE_COUNT)} hidden mines.'
    state: dict[str, Any] = {'spec': dict(spec), 'game': _new_game_from_spec(spec), 'goal': goal or _make_goal(spec), 'temperature': float(temperature), 'running': False}
    score_bar = widgets.HTML(value=_render_score_bar(state['game']))
    board_container = widgets.VBox([])
    cell_buttons: dict[tuple[int, int], widgets.Button] = {}
    mode_radio = widgets.RadioButtons(options=[('reveal', 'reveal'), ('flag', 'flag'), ('unflag', 'unflag')], value='reveal', description='Click mode:', layout=widgets.Layout(width='auto'))
    next_btn = widgets.Button(description='Next Move (LLM)', button_style='primary')
    auto_btn = widgets.Button(description='Auto-play 5', button_style='info')
    reset_btn = widgets.Button(description='Reset', button_style='warning')
    size_input = widgets.BoundedIntText(value=state['spec'].get('board_size', DEFAULT_BOARD_SIZE), min=2, max=12, step=1, description='Board:', layout=widgets.Layout(width='180px'))
    mines_input = widgets.BoundedIntText(value=state['spec'].get('mine_count', DEFAULT_MINE_COUNT), min=1, max=140, step=1, description='Mines:', layout=widgets.Layout(width='180px'))
    temp_input = widgets.BoundedFloatText(value=state['temperature'], min=0.0, max=2.0, step=0.05, description='Temp:', layout=widgets.Layout(width='200px'))
    newgame_btn = widgets.Button(description='New Game', button_style='success')
    trace_out = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', max_height='320px', overflow='auto'))

    def _rebuild_board() -> None:
        n = state['game'].board_size
        cell_buttons.clear()
        grid = widgets.GridspecLayout(n + 1, n + 1, grid_gap='2px', width=f'{(n + 1) * 46}px', height=f'{(n + 1) * 38}px')
        grid[0, 0] = widgets.HTML('')
        for c in range(n):
            grid[0, c + 1] = widgets.HTML(f'<div style="text-align:center; font-family:monospace; font-weight:bold;">{c + 1}</div>')
        for r in range(n):
            grid[r + 1, 0] = widgets.HTML(f'<div style="text-align:center; font-family:monospace; font-weight:bold;">{chr(ord('A') + r)}</div>')
            for c in range(n):
                desc, style = _cell_descriptor(state['game'], r, c)
                btn = widgets.Button(description=desc, button_style=style, layout=widgets.Layout(width='42px', height='34px', padding='0'))
                btn.on_click(_make_cell_handler(r, c))
                cell_buttons[r, c] = btn
                grid[r + 1, c + 1] = btn
        board_container.children = (grid,)

    def _refresh_cells() -> None:
        for (r, c), btn in cell_buttons.items():
            desc, style = _cell_descriptor(state['game'], r, c)
            btn.description = desc
            btn.button_style = style

    def _refresh() -> None:
        score_bar.value = _render_score_bar(state['game'])
        _refresh_cells()
        if state['game'].status != 'playing':
            next_btn.disabled = True
            auto_btn.disabled = True
            for btn in cell_buttons.values():
                btn.disabled = True
        else:
            next_btn.disabled = False
            auto_btn.disabled = False
            for btn in cell_buttons.values():
                btn.disabled = False

    def _do_llm_move() -> bool:
        if state['game'].status != 'playing':
            return False
        decider = decide_fn or (lambda g, s: _decide_one_move(g, s, state['goal'], model_name, endpoint, state['temperature']))
        observation, decision, source, raw_reply = decider(state['game'], state['spec'])
        state['game'], result = apply_decision(state['game'], decision)
        _append_trace_html(trace_out, state['game'].turn, decision, result, source, raw_reply)
        _refresh()
        return state['game'].status == 'playing'

    def _do_human_move(r: int, c: int) -> None:
        if state['game'].status != 'playing':
            return
        action = mode_radio.value
        allowed = state['spec'].get('allowed_actions', ['reveal', 'flag', 'unflag'])
        if action not in allowed:
            with trace_out:
                from IPython.display import HTML, display
                display(HTML(f'<div style="color:#c62828; font-style:italic; padding:4px;">✋ blocked: spec does not allow <b>{action}</b> (allowed_actions = {list(allowed)}). Change the click mode or update the spec.</div>'))
            return
        decision = {'action': action, 'args': {'row': chr(ord('A') + r), 'col': c + 1}, 'reason': f'human clicked {format_coord(r, c)} in {action} mode'}
        state['game'], result = apply_decision(state['game'], decision)
        _append_trace_html(trace_out, state['game'].turn, decision, result, 'human', None)
        _refresh()

    def _make_cell_handler(r: int, c: int) -> Callable:

        def _on_click(_btn) -> None:
            if state['running']:
                return
            _do_human_move(r, c)
        return _on_click

    def _on_next(_btn) -> None:
        if state['running']:
            return
        state['running'] = True
        next_btn.disabled = True
        auto_btn.disabled = True
        try:
            _do_llm_move()
        except Exception as exc:
            with trace_out:
                print(f'❌ error during LLM move: {exc}')
                traceback.print_exc()
        finally:
            state['running'] = False
            _refresh()

    def _on_auto5(_btn) -> None:
        if state['running']:
            return
        state['running'] = True
        next_btn.disabled = True
        auto_btn.disabled = True
        try:
            for _ in range(5):
                if not _do_llm_move():
                    break
                time.sleep(auto_play_pause)
        except Exception as exc:
            with trace_out:
                print(f'❌ error during auto-play: {exc}')
                traceback.print_exc()
        finally:
            state['running'] = False
            _refresh()

    def _start_new_game(label: str) -> None:
        state['game'] = _new_game_from_spec(state['spec'])
        trace_out.clear_output()
        _rebuild_board()
        _refresh()
        with trace_out:
            from IPython.display import HTML, display
            spec = state['spec']
            display(HTML(f'<div style="color:#666; font-style:italic;">— {label} ({spec.get('board_size', DEFAULT_BOARD_SIZE)}x{spec.get('board_size', DEFAULT_BOARD_SIZE)}, {spec.get('mine_count', DEFAULT_MINE_COUNT)} mines, temp={state['temperature']:.2f}, allowed={spec.get('allowed_actions', 'all')}) —</div>'))

    def _on_reset(_btn) -> None:
        _start_new_game('reset')

    def _on_new_game(_btn) -> None:
        new_size = int(size_input.value)
        new_mines = int(mines_input.value)
        if new_mines >= new_size * new_size:
            with trace_out:
                from IPython.display import HTML, display
                display(HTML(f'<div style="color:#c62828;">❌ mines ({new_mines}) must be less than {new_size}*{new_size}={new_size * new_size}.</div>'))
            return
        state['spec'] = {**state['spec'], 'board_size': new_size, 'mine_count': new_mines}
        state['goal'] = _make_goal(state['spec'])
        _start_new_game('new game')

    def _on_temp_change(change) -> None:
        state['temperature'] = float(change['new'])
    next_btn.on_click(_on_next)
    auto_btn.on_click(_on_auto5)
    reset_btn.on_click(_on_reset)
    newgame_btn.on_click(_on_new_game)
    temp_input.observe(_on_temp_change, names='value')
    _rebuild_board()
    _refresh()
    controls = widgets.HBox([next_btn, auto_btn, reset_btn])
    config = widgets.HBox([size_input, mines_input, temp_input, newgame_btn])
    return widgets.VBox([score_bar, board_container, mode_radio, controls, config, trace_out])


In [ ]:
from IPython.display import HTML, display
demo = MinesweeperGame(board_size=6, mine_count=6, seed=11)
demo.reveal(0, 0)
demo.flag(2, 2)
demo.flag(4, 1)
display(HTML(render_html(demo)))


### Try one move yourself

Below we start a fresh game and reveal the cell at row C, column 3 by hand — no agent, no LLM, just one direct call into the game engine. Watch what the board looks like before and after, and notice that flood-fill auto-opens the connected zero-cells around it.

In [ ]:
from IPython.display import HTML, display
manual = MinesweeperGame(board_size=6, mine_count=6, seed=7)
row, col = parse_coord('C3', manual.board_size)
result = manual.reveal(row, col)
print('Tool result:', result['message'])
display(HTML(render_html(manual)))


## Step 1: What Makes an Agent Different?

Before we touch the LLM, slow down for a minute. Three different kinds of programs could try to do our spec-fixing task. They look similar on the surface, but inside they behave very differently:

| Approach | What you will see | Where it falls short |
|---|---|---|
| **Traditional Code** | A fixed rule a programmer wrote ahead of time | Cannot explain alternatives; rewriting needed when the goal changes |
| **Chatbot** | A natural-language answer | Gives advice but never inspects or changes the actual spec |
| **Agent** | A loop: observe, choose, act, check, revise | Needs clearly defined tools, constraints, and safety boundaries |

I want you to feel each row. Traditional code is the calculator you already trust. The chatbot is the friend who talks well but never picks up a pen. The agent picks up the pen *and* shows you their work.

### Key terms

We will use these words the whole way through:

- **Goal** — the outcome we are after (today: a playable Minesweeper with the right toolkit).
- **Constraint** — a rule the answer must satisfy (here: total complexity ≤ budget AND every required type present).
- **Tool** — a normal Python function the agent is allowed to call.
- **Observation** — what the agent notices when it looks at the current spec or board.
- **Candidate action** — one possible next step.
- **Reflection** — the follow-up check that asks "did that actually help?"
- **Reasoning trace** — a visible, readable record of how the agent decided.

If you only take one sentence away from this whole notebook, take this one:

```text
Prompt → Response

becomes

Goal → Observe → Plan → Tool Use → Check → Reflect → Revise → Result
```

Every step from here on is one piece of that second line.

### What is an Agent, in one sentence?

> An **Agent** is a piece of software that keeps a goal in mind, observes the current situation, picks an action, calls a tool, and then checks whether that action helped.

I did not say "the AI thinks." That phrase is too vague to teach with. A picture I find more useful is the **lab experiment loop**:

```text
Goal → measure current state → choose an intervention → apply it → measure again
```

In our notebook:

- **Goal** = a playable Minesweeper that clears every non-mine cell.
- **Measurement** = checking the spec's complexity and required types (Phase A), or checking the board state (Phase B).
- **Intervention** = adding/removing/simplifying a spec feature (Phase A), or revealing/flagging a cell (Phase B).
- **Second measurement** = was the constraint repaired? did the move keep us alive?

So an agent is not magic, and it is not pretending to think. It is just a visible control loop wrapped around a task.

### A mental model you can carry with you

Anytime someone says "agent," ask yourself: where is the **goal**, where is the **state**, where are the **tools**, where is the **feedback**, and what does **revision** look like? If you can name all five, it is an agent.

That is why a game-builder + game-player combo is so useful — every piece shows up twice, once in each phase:

| Concept | Phase A (design) | Phase B (play) |
|---|---|---|
| Goal | "Spec with complexity ≤ budget and all required types" | "Reveal every non-mine cell" |
| State | The `spec` dictionary | The board (revealed/flagged/hidden cells) |
| Tool | `add_feature` / `remove_feature` / `simplify_feature` | `reveal` / `flag` / `unflag` (whichever Phase A allowed) |
| Feedback | "complexity 11 exceeds limit 5" / "missing 'safety'" | "BOOM at C4" or "revealed 4 cells" |
| Revision | Add a feature, drop a feature | Reveal a different cell next turn |

Keep this table next to you. You will see how every step we take maps to one of these five words — in one of two phases.

### Two words you will see a lot: `spec` and `complexity`

Before we open our first broken spec, let me quickly nail down two terms that show up over and over for the rest of the notebook.

- **`spec`** — a plain Python dictionary that describes the Minesweeper game we want to build. It lists every feature plus the rules the finished game must satisfy. The fields you will see in Step 3:
  - `features` — the list of things the game will have, e.g. `"Reveal cell"`, `"Flag cell"`, `"Numeric hints (1-8)"`. Each feature has a `type`, a `complexity`, and a `status` (active/removed).
  - `complexity_limit` — the total budget the spec is allowed to spend (we start with **5**).
  - `must_include` — the types of features that **must** appear in the active list, e.g. `["action", "hint", "safety"]`.
  - `board_size`, `mine_count`, `first_click_safe` — straight game parameters that go directly into Phase B without the agent touching them.
- **`complexity`** — a small integer "cost" we attach to every feature. Cheap features cost 1; expensive ones cost 2 or 3.
  - The sum of `complexity` over all *active* features must stay ≤ `complexity_limit`.
  - It is a teaching device, not a real engineering metric. We use it so the agent has a concrete budget to negotiate against — every "remove" or "simplify" the agent picks visibly shrinks the total.

So when Step 3 prints `complexity 11 exceeds limit 5`, read it as: *"the seven active features add up to 11 cost points, but the budget is 5 — something has to be removed or simplified before the spec is acceptable."*

## Step 2: Import Libraries and Pin a Few Switches

Let us pull in everything we need and pin a few switches at the top. If we change anything later, you will know exactly where to look.

- `MAX_AGENT_STEPS = 5` — the safety belt for Phase A's spec-fixing loop.
- `REQUIRE_APPROVAL = True` — reminder for Phase A: never silently apply an important change.
- `BOARD_SIZE`, `MINE_COUNT`, `COMPLEXITY_LIMIT` — **the three knobs that change the broken-spec scenario**. Edit them in the next cell and rerun the notebook from Step 3 to make the puzzle easier or harder.
- `TEMPERATURE` — **the LLM creativity dial** used by both Phase A (spec fixing, Step 15) and Phase B (live play). Lower values (0.0–0.3) make the agent boring and predictable but reliable. Higher values (0.6–1.0) give visible variation between runs at the cost of occasional weird moves.
- `PRESET_GOALS` — three design briefs you can swap between later. We use the first for the rest of the notebook.

In [ ]:
from copy import deepcopy
from typing import Any
import pandas as pd
MAX_AGENT_STEPS = 5
REQUIRE_APPROVAL = True
BOARD_SIZE = 6
MINE_COUNT = 6
COMPLEXITY_LIMIT = 5
TEMPERATURE = 0.7

def make_initial_spec():
    """Build the starting spec with the user-tunable overrides applied."""
    s = create_initial_spec()
    s['board_size'] = BOARD_SIZE
    s['mine_count'] = MINE_COUNT
    s['complexity_limit'] = COMPLEXITY_LIMIT
    return s
PRESET_GOALS = ['Design a Minesweeper spec that fits within COMPLEXITY_LIMIT points and includes an action, a hint, and a safety feature.', 'Design a Minesweeper variant where every active feature is required and complexity is at most COMPLEXITY_LIMIT - 1.', 'Design a Minesweeper for absolute beginners: first-click safe, numeric hints, and at most one action besides reveal.']
goal = PRESET_GOALS[0]
print('Current goal:')
print(goal)
print(f'\nTunable knobs: BOARD_SIZE={BOARD_SIZE}, MINE_COUNT={MINE_COUNT}, COMPLEXITY_LIMIT={COMPLEXITY_LIMIT}, TEMPERATURE={TEMPERATURE}')


## Step 2.5: Ask LM Studio Which Models Are Loaded

Before any LLM step works, we need to talk to LM Studio and pick a model. Hardcoding a model name does not travel between your laptop, mine, and the lab — so we do what the Art Director case does and *ask* LM Studio what is loaded.

LM Studio exposes an OpenAI-compatible endpoint at `http://127.0.0.1:1234/v1/models`. A simple `GET` returns the list of currently-loaded models. The cell below:

1. Calls `list_lm_studio_models()`.
2. Prints every model id.
3. Picks the first one as `MODEL_NAME`. Override it in the next cell if you want a specific model.

If LM Studio is not reachable, this cell raises a `RuntimeError` with instructions. That is on purpose: every later cell depends on the LLM.

In [ ]:
available_models = list_lm_studio_models()
print(f'LM Studio reports {len(available_models)} model(s) loaded:')
for index, model_id in enumerate(available_models, start=1):
    print(f'  {index}. {model_id}')
MODEL_NAME = pick_default_model(available_models)
print(f'\nUsing default model: {MODEL_NAME}')
print(f'Endpoint: {DEFAULT_LM_STUDIO_ENDPOINT}')


## Step 3: Build the Spec — Phase A starts here

Every agent needs something to *look at*. In Phase A the environment is just a Python dictionary that describes the Minesweeper we want to build. Spec is fully visible; you can print it any time.

The spec you are about to see is deliberately broken in two ways at once:

- **Total complexity = 11**, but our budget is **5**. The spec is way over budget.
- The `must_include` list demands `["action", "hint", "safety"]`. The starting spec has `action` and `hint`, but you will not find a single feature with `type="safety"`.

Those two gaps are exactly what we want — they give the agent two concrete problems to detect and repair, and they give you two concrete things to watch get fixed.

When you run the cell, read the table top to bottom. Which features look essential? Which ones look like nice-to-haves? If *you* were to cut, which one would you cut first?

### You are playing two roles today

For Phase A you are not just a programmer — you are also the **game designer**. That second role matters: it is how you tell whether the agent is making sensible cuts. Then in Phase B you become the **director** who clicks `[Next Move →]` and watches the agent's reasoning unfold.

Think of how you would design a small game on paper:

1. Sketch a feature list.
2. Check whether it fits the complexity budget.
3. Notice missing required modules.
4. Decide what to cut, what to simplify, and what to add.
5. Check again.

That five-step routine is *already* an agent loop. All we do in Phase A is write it explicitly in Python so we both see it run:

| What you would do as a designer | Agent concept | In code |
|---|---|---|
| Sketch the feature list | State | the `spec` dictionary |
| Add up complexity | Observation | `check_complexity(spec)` |
| Compare possible cuts | Planning | the `candidates` list |
| Edit the spec | Tool use | `remove_feature(...)` |
| Check the revised draft | Reflection | `observe(revised_spec)` |

### Why we start with state

For an agent, the **state** is the worksheet it can actually look at. Without state, there is nothing to observe — and an agent with nothing to observe is a chatbot in a costume.

We render the spec as a pandas table on purpose: I want you to see that Agentic AI is not magic hidden inside the model. It is a loop around a state you can read, point at, and edit at any time.

Before you run the next cell, predict for yourself:

- What is the complexity budget?
- Which feature types are required?
- Which features look most expensive (high complexity, optional)?

In [ ]:
def spec_to_dataframe(spec: dict[str, Any]) -> pd.DataFrame:
    return pd.DataFrame(spec['features'])
spec = make_initial_spec()
print(f'Title: {spec['title']}')
print(f'Audience: {spec['audience']}')
print(f'Board: {spec['board_size']}x{spec['board_size']} with {spec['mine_count']} mines')
print(f'Complexity budget: {spec['complexity_limit']}')
print(f'Required types: {spec['must_include']}')
print(f'Active complexity total: {total_complexity(spec)} (over budget!)')
spec_to_dataframe(spec)


## Step 4: Define Tools

A **tool** in our world is an ordinary Python function. I want to say that out loud because the word "tool" gets a lot of mystique in AI demos that it does not earn.

Here is the first big agent idea: the LLM does not change the spec directly. It can only pick from the tools *we* give it. That is not a limitation — it is a safety property we lean on later when LM Studio drives the loop.

Two kinds of tools:

1. **Check tools** — read the spec and report a constraint result. Never modify anything.
2. **Action tools** — return a *revised* spec. Never mutate the old one in place.

Notice the second point. Every action tool returns a brand-new spec. That is deliberate: you can always compare "before" and "after," and a buggy action cannot silently corrupt state mid-run.

### Tools are the bridge from words to action

People sometimes say "if the LLM is smart enough, it can just do the task." I want to push back on that. An LLM, on its own, only produces text. To make anything happen outside the chat — change a file, edit a spec, send an email — *something* has to convert text into a real function call. In big systems we call those somethings "tools."

The key idea: **an agent does not replace your normal code. It organizes your normal code around a goal.** The tools below are exactly the kind you would write for any small Python project:

- `check_complexity()` reads the spec and tells us whether the total fits the budget.
- `check_requirements()` reads the spec and tells us whether every required type is present.
- `remove_feature()`, `simplify_feature()`, `add_feature()` change the spec and hand back a revised copy.

Five small functions. That is the entire toolbox the agent uses in Phase A.

In [ ]:
print('Complexity check:', check_complexity(spec))
print('Requirements check:', check_requirements(spec))


## Step 5: Try the Action Tools

The three action tools all live in the self-contained helper cell. We already imported them. Let us call one by hand so you see exactly what a tool call looks like before the agent starts orchestrating.

After this cell:

- `trial_spec` is what the spec would look like if we dropped the largest non-required feature in one move.
- `tool_result` is the small dict the agent will read when it asks "did that work?"

We are not committing this change — `spec` itself is untouched. Immutability promise in action.

In [ ]:
trial_spec, tool_result = remove_feature(spec, 'Auto-flag completed')
print('Tool result:', tool_result)
print('After tool call — complexity check:', check_complexity(trial_spec))
spec_to_dataframe(trial_spec)


## Steps 6-7: Two Baselines Before the Agent

Before the agent takes over, run the two simpler approaches on the SAME broken spec so you have something to compare against. Keep the Step 3 spec table in view.

| Approach | What it does here | What it cannot do |
| --- | --- | --- |
| **Traditional code** | One fixed rule: if over budget, drop the most expensive optional feature | Cannot explain a choice, compare options, or even *notice* the missing `safety` requirement |
| **Chatbot** | Describes a nice design in words | Never inspects or changes the actual spec — the state is untouched |
| **Agent** (next steps) | Observes, plans, acts, checks, and revises in a loop | Needs defined tools, constraints, and safety boundaries |

Run the next two cells and watch the spec table: the fixed rule changes the numbers but misses `safety`; the chatbot talks well but changes nothing.

In [ ]:
traditional_spec = make_initial_spec()
traditional_result = None
if check_complexity(traditional_spec)['total'] > traditional_spec['complexity_limit']:
    optional = [f for f in traditional_spec['features'] if not f['required'] and f['status'] == 'active']
    if optional:
        biggest = max(optional, key=lambda f: f['complexity'])
        traditional_spec, traditional_result = remove_feature(traditional_spec, biggest['name'])
print('Traditional code result:', traditional_result)
print('Complexity check:', check_complexity(traditional_spec))
print('Requirements check:', check_requirements(traditional_spec), '← still missing!')
spec_to_dataframe(traditional_spec)


### And now a chatbot on the same task

We call the LM Studio model you picked in Step 2.5. Watch the spec table as it answers.

In [ ]:
chatbot_text = chatbot_baseline(goal, model=MODEL_NAME)
print('--- Chatbot reply ---')
print(chatbot_text)
print('\n--- Spec after chatbot reply ---')
print('Notice: the spec has not changed at all.')
print('Complexity check:', check_complexity(spec))
print('Requirements check:', check_requirements(spec))


## Step 8: Observe — the First Step of the Loop

OK, enough comparison. Let us start building the agent.

The very first thing our agent will do — before *acting* — is **observe**. It looks at the current spec, runs the check tools, and writes down what it found. No action yet, just measurement.

Think of it like a lab measurement. Before changing an experiment, you measure what is there. Same idea here: before changing the spec, the agent measures the spec.

### A simple state machine inside the agent

If "loop" still feels abstract, here is a picture. At any given moment our agent is in one of five states:

```text
OBSERVING  → looking at the current spec
PLANNING   → listing candidate actions
ACTING     → calling one tool
REFLECTING → checking whether that tool actually worked
DONE       → all constraints pass; we can stop
```

Written out as Python, the whole loop fits in about twenty lines. If you have ever written a small game or a turn-based puzzle, this should feel familiar — it is a loop that updates after every piece of feedback. That is all "agentic" really means in this notebook.

### Observation = measurement, no more, no less

Before acting, the agent asks three small questions:

- What is the total complexity right now?
- Are any required feature types missing?
- Did any rule fail?

`observe()` returns a structured dictionary instead of just printing a message. The *next* step — planning — needs to read those answers and reason about them. If observation were just a print, planning would be flying blind.

In [ ]:
agent_spec = make_initial_spec()
observation = observe(agent_spec)
print('Observation:')
for key, value in observation.items():
    print(f'  {key}: {value}')


## Step 9: Plan — List the Candidate Actions

We have observed; now we plan. This is where the agent stops behaving like fixed code.

Instead of jumping straight to a single hard-coded action, the agent lays out *several* candidate moves. Each candidate carries:

- a **tool name** — which function to call?
- the **arguments** — with what inputs?
- a **score** — how good we think this option is.
- a **reason** — written in plain English so you can read it.

When the candidate table appears, slow down and read each row. This is where you can really *feel* the difference between fixed code and an agent.

### Planning means comparing possible next moves

For our broken spec the menu will include both:

- **An add-feature move** for the missing `safety` type — without it the `must_include` constraint stays broken.
- **One or more remove-feature moves** for the largest non-required features (Auto-flag completed costs 3, Mine counter HUD and Flood-fill cost 2 each).

The `score` column is not "real intelligence." It is a classroom-friendly way to make decision criteria visible. The score is just a hook you can grab onto when you ask yourself:

- Which option makes a failing constraint pass?
- Which option keeps the most important features intact?
- Which option leaves Phase B with the most useful tool set?

When you stop and discuss those questions out loud, *that* is the real learning moment of this notebook.

In [ ]:
candidates = propose_candidate_actions(agent_spec, observation)
print(f'{len(candidates)} candidate action(s):')
pd.DataFrame(candidates)


## Step 10: Act — Pick One and Run It

We have observed; we have planned. Now we act.

In this step the agent picks the candidate with the highest score, looks up the matching Python function in `TOOLS`, and calls it. That call is the moment the agent stops being "just text" and changes the spec for real.

### Acting means calling a real tool

The action you pick becomes a tool call. The instant that call returns, the agent has *done* something — not just talked about doing it.

The `TOOLS` dictionary from the self-contained helper cell maps a tool name like `remove_feature` to the actual Python function. The mapping is deliberately small. The agent *cannot* invent functions; if `remove_feature` is not in `TOOLS`, the agent cannot call it. Period.

That tiny dictionary is also our first safety lesson: good agent design means giving the system *useful* tools, not unlimited power.

In [ ]:
print('Tool surface the agent can use in Phase A:')
for name in TOOLS:
    print(f'  - {name}')
selected_action = choose_action(candidates)
print(f'\nSelected action: {selected_action['label']}')
print(f'  tool : {selected_action['tool']}')
print(f'  args : {selected_action['args']}')
print(f'  score: {selected_action['score']}')
print(f'  reason: {selected_action['reason']}')
revised_spec, action_result = act(agent_spec, selected_action)
print(f'\nTool result: {action_result}')
spec_to_dataframe(revised_spec)


## Step 11: Reflect — Did It Actually Work?

The fourth step is **reflection**. Sounds philosophical, is not — reflection just means: after you act, measure again.

Without this step the agent would have to *assume* its action worked. That is a great way to build a system that confidently lies to you. With reflection, the agent faces the new spec and admits when its move did not fix the problem.

### Reflection is what closes the loop

If you have ever debugged a program, you already do this:

```text
Hypothesis → action → measurement → revise if needed
```

That is literally the scientific method, and it is what the agent is doing here. If the revised spec still fails a check, the agent loops back to planning and picks a different candidate. *That* is what turns "single LLM call" into "agent loop."

In [ ]:
reflection = observe(revised_spec)
print('Reflection on the revised spec:')
for key, value in reflection.items():
    print(f'  {key}: {value}')


## Step 12: Build a Reasoning Trace

So far the agent has observed, planned, acted, and reflected. Now we capture all of that in one readable **reasoning trace**.

A trace is a print-friendly story of what the agent did. In this notebook the trace matters more than the final answer. The answer is one piece of data; the trace tells you *why* that answer is the one we ended up with.

### Why we always show the trace

A good trace shows five things:

1. What the agent **noticed** (observation).
2. What options it **considered** (candidates).
3. **Why** it picked the one it picked.
4. Which **tool** it actually called.
5. Whether the new spec **passed** the next check.

This is not only useful for teaching. In production systems, traces are exactly how engineers debug agents and how safety teams audit them.

In [ ]:
def render_trace(trace: dict[str, Any], goal: str) -> str:
    lines = [f'🎯 Goal: {goal}', f'👀 Observation: {('; '.join(trace['observation']['issues']) if trace['observation']['issues'] else 'all constraints pass')}', '🧠 Candidate Decisions:']
    for index, candidate in enumerate(trace['candidates'], start=1):
        lines.append(f'  {index}. {candidate['label']} | score={candidate['score']:.2f} | {candidate['reason']}')
    action = trace['action']
    lines.extend([f'🛠 Tool Call: {action['tool']}({action['args']})' if action else '🛠 Tool Call: none', f'📌 Result: {trace['result']['message']}', f'✅ Reflection: {('all constraints met' if trace['reflection']['passed'] else '; '.join(trace['reflection']['issues']))}'])
    output = '\n'.join(lines)
    print(output)
    return output
trace = {'observation': observation, 'candidates': candidates, 'action': selected_action, 'result': action_result, 'reflection': reflection}
_ = render_trace(trace, goal)


## Step 13: Human Approval — Pause Before Important Changes

Human approval is one of the most underrated parts of agent design. It is also one of the simplest.

A classroom agent should not silently apply an important change. Before the agent commits its choice, we show you the recommendation and let you decide: Approve, Modify, or Replan.

Small code, but the difference between an agent you can confidently demo and one you cannot.

### Approval is part of the design, not a patch

A good classroom agent is powerful enough to act, *but* bounded enough to be reviewed. The approval gate teaches two ideas:

- An agent **can** automate multi-step work and save you time.
- Important decisions **should** still be visible, and a human should be able to step in.

The same pattern has a name in production systems: **human-in-the-loop**.

In [ ]:
approval = 'Approve'
print(f'Human approval choice: {approval}')
if approval == 'Approve':
    approved_spec = revised_spec
else:
    approved_spec = agent_spec
spec_to_dataframe(approved_spec)


## Step 14: The Final Mission Report

Now that the agent (and you) have agreed on a revised spec, we wrap things up with a small **final report**. It translates the internal state into a friendly summary — title, board size, mine count, total complexity, whether the checks passed, and **which actions Phase B will have available**.

That last field is the bridge from Phase A to Phase B. Watch it closely — every choice you (or the agent) made about removing features is now baked into the toolset Phase B's player will use.

### Before you look at the report — pause

Cover up the report and try to answer:

1. What were the two original problems with the spec?
2. Which candidate actions did the agent compare?
3. Why did it pick *that* one instead of another?
4. Which specific tool actually changed the spec?
5. Which Phase B actions will be available given the surviving features?

If you can answer all five, you understand Phase A.

In [ ]:
report = generate_final_report(approved_spec)
print('=== Final mission report ===')
for key, value in report.items():
    if key == 'active_features':
        print(f'{key}:')
        for item in value:
            print(f'  - {item['name']} (complexity={item['complexity']}, type={item['type']})')
    else:
        print(f'{key}: {value}')


## Step 15: Hand the Phase A Loop to LM Studio

So far every "decision" you saw in Phase A was made by deterministic Python: we hand-built `propose_candidate_actions` and picked the highest score. That is great for teaching the *shape* of an Agent loop, but it does not yet show what changes when a real LLM is in the driver seat.

In this step we do exactly that for Phase A. The Python tools, observation function, and candidates stay exactly as they were. The only thing that changes is **who chooses the next action**: instead of `max(candidates, key=score)`, we send the current observation + candidate list to LM Studio and ask it to reply with a single JSON tool call. Then we run that tool, observe again, send the new state back, and repeat until the constraints pass or the LLM says `stop`.

Each iteration is one round-trip with LM Studio. Read each step like a conversation. If LM Studio briefly misbehaves on a single round, the loop transparently falls back to the deterministic best candidate so the demo always finishes — you will see `(fallback)` in the source column when that happens.

In [ ]:
def print_llm_step(trace: dict[str, Any]) -> None:
    print(f'\n--- LLM Agent Step {trace['step']} ({trace['source']}) ---')
    issues = trace['observation']['issues'] or ['all constraints pass']
    print(f'👀 Observation: {'; '.join(issues)}')
    if trace['llm_raw']:
        print(f'🤖 LM Studio reply: {trace['llm_raw'].strip()[:240]}')
    action = trace['action']
    if action:
        print(f'🛠 Tool Call: {action['tool']}({action.get('args', {})})')
        print(f'   Reason: {action.get('reason', '')}')
    result = trace['result']
    if result:
        print(f'📌 Result: {result['message']}')
    reflection_issues = trace['reflection']['issues'] or ['all constraints met']
    print(f'✅ Reflection: {'; '.join(reflection_issues)}')
llm_final_spec, llm_traces = llm_agent_loop(initial_spec=make_initial_spec(), goal=goal, observe_fn=observe, propose_fn=propose_candidate_actions, tools=TOOLS, max_steps=MAX_AGENT_STEPS, model=MODEL_NAME, on_step=print_llm_step, temperature=TEMPERATURE)
print('\nFinal spec after the LLM-driven loop:')
spec_to_dataframe(llm_final_spec)


In [ ]:
print('=== LLM-driven final report ===')
llm_report = generate_final_report(llm_final_spec)
for key, value in llm_report.items():
    if key == 'active_features':
        print(f'{key}:')
        for item in value:
            print(f'  - {item['name']} (complexity={item['complexity']}, type={item['type']})')
    else:
        print(f'{key}: {value}')
assert llm_report['complexity_passed'], 'LLM loop did not fix complexity'
assert llm_report['requirements_passed'], 'LLM loop did not fix requirements'
print()
print(">>> Phase A complete. The agent's choice in Phase A directly shapes Phase B's toolset:")
print(f'>>>   Phase B allowed_actions = {llm_report['allowed_actions_for_play']}')
if 'flag' not in llm_report['allowed_actions_for_play']:
    print(">>> Notice: 'flag' is NOT in Phase B's toolset because Phase A removed 'Flag cell'.")
    print('>>> The Phase B agent will have to win using only reveal — a real consequence of design.')


In [ ]:
phase_b_actions = list(llm_report['allowed_actions_for_play'])
print(f'Phase B will use: {phase_b_actions}')


## Phase B: Now Let the Agent Actually Play the Game

This is the moment the case earns its name. We hand the Agent the spec it just finalised, and we sit back and watch it play Minesweeper one click at a time. You drive each turn with `[Next Move →]`.

There are three things to keep your eye on:

1. **The board redraws** every turn after the LLM picks a move.
2. **The trace area** shows what the LLM saw, what JSON it returned, what tool we ran, and what happened.
3. **The toolset is whatever Phase A left us with.** Removed `Flag cell` in Phase A? The agent literally cannot flag in Phase B. Removed `Unflag cell`? It cannot take a flag back. Welcome to Spec → Behaviour causality.

When you click `[Next Move →]`, the cell does this for you:

```python
observation = observe(game, spec)              # ASCII board + JSON state
candidates  = propose_minesweeper_actions(game, spec)
decision    = ask LLM for one JSON tool call   # one round-trip
game        = apply_decision(game, decision)
refresh widget + append trace
```

If the LLM is slow or returns garbage on a single round, the click handler falls back to the deterministic best candidate so the demo never freezes. You will see `[fallback]` in the trace when that happens.

### First, see the model with no candidate list

Phase A's play loop (and the widget below) hands the model a **candidate list** — a few pre-validated moves it can copy. Before you rely on that safety net, look at what a small model does *without* it: we send only the board and ask for a move. Watch the raw reply. Is it valid JSON? One action or several? Are the coordinates even on the board? This is exactly the failure the candidate list is designed to absorb.

In [ ]:
demo_game = MinesweeperGame(board_size=4, mine_count=3, seed=7)
demo_game.reveal(0, 0)
print('Board the model sees:')
print(observe_for_llm(demo_game)['board_ascii'])
raw_reply, parsed = ask_free_move(demo_game, model=MODEL_NAME, temperature=TEMPERATURE)
print('\n--- Raw model reply (free generation) ---')
print(raw_reply)
print('Parsed as JSON:', parsed)
print('\n--- Python candidate actions (what the widget uses) ---')
for cand in propose_minesweeper_actions(demo_game):
    print(f'  {cand['tool']} {cand['args']}  score={cand['score']:.2f}  {cand['reason']}')


### Read the live widget like a transcript

Every entry in the trace has the same shape:

```
Turn N  [llm|fallback]  🤖 action at <row><col>
        💭 the agent's one-sentence reason
        📌 result message ("flagged C3" or "revealed 4 cells starting at B5" or "BOOM at A1")
```

If you click the "show LLM raw reply" disclosure, you can see the literal JSON the model returned. That is the same content the next turn's prompt remembers.

If you want to start over with the same spec, click `[Reset]`. If you want to step through 5 moves quickly, click `[Auto-play 5 moves]`.

Run the cell below to spawn the live game widget.

In [ ]:
play_spec = {**llm_final_spec, 'allowed_actions': phase_b_actions, 'first_click_safe': True, 'flood_fill': True, 'seed': 42}
minesweeper_ui = build_minesweeper_widget(spec=play_spec, model_name=MODEL_NAME, temperature=TEMPERATURE)
minesweeper_ui


## Step 17: Run It Again with a Different Seed

Same spec, different mine layout. The LLM has to reason from scratch every game — there is no transferable strategy between boards beyond the deduction rules in the system prompt.

Click `[Auto-play 5 moves]` on each widget to advance them in batches. Compare:

- How many turns does each game last?
- How often does the agent guess vs. deduce?
- Are the `[fallback]` rounds correlated with hard board states?

In [ ]:
play_spec_alt = {**play_spec, 'seed': 123}
minesweeper_ui_alt = build_minesweeper_widget(spec=play_spec_alt, model_name=MODEL_NAME, temperature=TEMPERATURE)
minesweeper_ui_alt


## Step 18: Now It Is Your Turn — Change the Spec

Three small experiments I would love you to try before closing the notebook:

1. **Strip the agent's flag tool.** Replace `play_spec["allowed_actions"]` with `["reveal"]` and rerun the widget. The agent now cannot flag at all. Does its win rate drop? Does the trace look different?
2. **Give the agent extra hint depth.** Lower `mine_count` to 3 on a 6×6 board so deductions are easier — does the agent finish in fewer turns?
3. **Change `seed`** and run several games. Note when the agent guesses (corner reveal) vs deduces (flag based on a numeric clue).

Each of these is one small edit to the spec, then one rerun. The whole point of the loop is that the same code keeps running — the spec changes, the agent's behaviour changes, the code does not.

The cell below shows the "no flag" experiment. Click `[Auto-play 5 moves]` a couple of times and see what happens.

In [ ]:
experiment_spec = {**play_spec, 'allowed_actions': ['reveal'], 'seed': 99}
experiment_ui = build_minesweeper_widget(spec=experiment_spec, model_name=MODEL_NAME, temperature=TEMPERATURE)
print(f'Experiment: allowed_actions = {experiment_spec['allowed_actions']}')
print('The agent cannot flag in this game. Watch how its trace changes.')
experiment_ui


## Bonus: A Fully Autonomous Agent (No Candidate List)

The widget above always chooses from a Python-generated candidate list, and falls back to the best one if the model stumbles. Here we remove that safety net and let the model drive the whole game itself. A candidate-free agent needs three guardrails you can watch working:

1. **Action history** — every past move (success *and* failure) goes back into the prompt, so the model stops re-picking the same cell.
2. **Retry with feedback** — if the reply is not valid JSON or the coordinate is off the board, we hand the error back and ask again (up to a small budget).
3. **Validation before acting** — only a well-formed, on-board move is applied; the game engine still decides whether it was actually useful.

Run it against a real model (LM Studio) and read the trace. You will usually see it make progress, occasionally retry a malformed reply, and — because Minesweeper has real luck — sometimes guess into a mine.

In [ ]:
def print_autonomous_step(trace: dict[str, Any]) -> None:
    RED, GREEN, RESET = ('\x1b[91m', '\x1b[92m', '\x1b[0m')
    print(f'\n--- Autonomous Step {trace['step']} (retries={trace['retries']}) ---')
    print('Board before:')
    print(trace['board_ascii'])
    move = trace['decision']
    if move:
        args = move.get('args', {})
        print(f'        ↓  {move.get('action')} {args.get('row')}{args.get('col')} — {move.get('reason', '')}')
    else:
        print('        ↓  (no valid move produced)')
    print('Board after:')
    print(trace['board_after'])
    result = trace['result']
    if result:
        if result.get('success'):
            print(f'{GREEN}✔ Result [OK]: {result.get('message', '')}{RESET}')
        else:
            print(f'{RED}⚠ Result [FAILED]: {result.get('message', '')}{RESET}')
free_game = MinesweeperGame(board_size=4, mine_count=3, seed=11)
autonomous_traces = autonomous_agent_loop(free_game, model=MODEL_NAME, temperature=TEMPERATURE, max_steps=12, max_retries=2, on_step=print_autonomous_step)
print(f'\nFinished: status={free_game.status}, steps taken={len(autonomous_traces)}')


### Candidate-constrained vs fully autonomous

| Aspect | Candidate list (the widget) | Fully autonomous (above) |
| --- | --- | --- |
| Where moves come from | Python deterministic rules propose; the model selects | The model reads the board and produces the move itself |
| Invalid moves | The list only contains legal cells — none appear | Possible; the action history and retries correct them |
| Format stability | The model copies a candidate's JSON — almost never breaks | JSON can fail to parse; a retry recovers it |
| Reasoning quality | A 0.95 "flag" candidate is certainly correct | The model may deduce correctly or misjudge and hit a mine |
| Safety net | Falls back to the best candidate | Retry + error feedback + history; no free fallback |

Both share the same core loop — **Observe → Think → Act → Read Feedback**. The only difference is how much freedom the *Think* step has, and where the safety boundary sits. Constrained is steadier for a classroom; autonomous shows what real guardrail engineering has to handle.

## Summary and Key Learnings

Congratulations! You did not just *read* about an agent — you watched one fix a broken Minesweeper spec, then watched the SAME agent play the game it just finalised. The big shift we covered:

```text
Prompt → Response
```

becomes

```text
Goal → Observe → Plan → Tool Use → Check → Reflect → Revise → Result
```

### 1. Agentic AI fundamentals
- An Agent keeps a goal in mind, observes state, picks a tool, runs it, checks the result.
- The "agentic" part is the *loop*, not any single LLM call.

### 2. Tool use
- LLMs only produce text. To change the world we wire them to ordinary Python functions.
- Phase A used three spec-editing tools. Phase B used three game-playing tools. Same loop shape both times.

### 3. Reasoning traces
- A trace is the visible record of what the Agent observed, considered, did, and concluded.
- In this notebook, the trace mattered more than the final answer — it was the teaching artifact.

### 4. Human approval
- A safe classroom agent never silently applies an important change.
- The Approve / Modify / Replan gate is small code but big design.

### 5. A real LM Studio loop
- The same Observe → Plan → Act → Reflect skeleton works when an LLM picks the next action.
- We discovered the model with `GET /v1/models`, not by hard-coding.
- A transparent fallback keeps the demo running if LM Studio hiccups on a single round.

### 6. Spec → Behaviour causality (the headline lesson)
- The choice you made in Phase A — "remove Flag cell to fit the budget" — literally removed `flag` from the Phase B agent's toolset.
- Watching the Phase B agent struggle without a tool you removed in Phase A is the most concrete possible demonstration that **agent design choices show up as agent behaviour later**.

### 7. From spec to playable game
- The widget literally cannot give the agent a tool the spec did not enable. Strip `flag` from `allowed_actions`, and the agent's candidate list and JSON action set both shrink.
- The spec is not decoration — it is what makes the agent's role real.

### Next Steps

- Try a different LM Studio model and compare how each one explains its reasoning.
- Add a new tool (for example, `chord` for auto-reveal) to both the self-contained game engine and the spec's `must_include`. Update the play prompt so the LLM knows about it.
- Try the whole notebook on a 4×4 board with 3 mines. Phase A converges in 1-2 steps; Phase B is a quick playable demo for a 10-minute class.

The All helper implementations are included in this notebook. Once comfortable, copy and adapt the relevant functions for different goals, tools, or games while keeping the same loop.